## 1. 필요 라이브러리 호출

In [3]:
# 환경설정
import os
import sys
import time
from tqdm import tqdm
import nest_asyncio
nest_asyncio.apply()
from dotenv import load_dotenv

load_dotenv()
# duckdb
import duckdb

# 데이터 전처리
import re
import pandas as pd
import numpy as np
import polars as pl
from datetime import datetime, timedelta
from copy import deepcopy

# 데이터 수집
import requests
from bs4 import BeautifulSoup


# VectorDB 저장
from hashlib import md5
from langchain_community.vectorstores.utils import filter_complex_metadata # ChromaDB가 제공하지 못하는 데이터 형태를 자동으로 string처리
from datetime import datetime, timezone

## LLM 활용
#from summary_function import NewsSummaryAgent
# LLM 활용을 위한 dict형태 구축
from collections import defaultdict

# langchain 계열
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

from langchain_openai import ChatOpenAI

# 1. LLM 모델 세팅 (OpenAI)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
#from summary_function_openai import NewsSummaryAgent, build_summary_graph  # 너가 만든 것
from summart_function_openai_2 import NewsSummaryAgent, build_summary_graph  # 너가 만든 것
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# ㄱRe-ranker 모델 활용
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder


## 2. ETF 목록 가져오기

In [8]:
ETF_conn = duckdb.connect('../DB/ETF.db')

In [9]:
ETF_df = ETF_conn.execute('select * from IRP_ETF_COMPOSE_table').fetchdf()
ETF_conn.close()

In [10]:
ETF_df = ETF_df.map(lambda x : x.strip())

In [11]:
ETF_df_task_1 = ETF_df[ETF_df['구성종목 종목명'] != "설정현금액"]
ETF_df_task_1 = ETF_df_task_1[ETF_df_task_1['구성종목 종목명'] != "원화현금"]
ETF_df_task_1 = ETF_df_task_1[1:]

In [12]:
# 원하는 컬럼 필터링
ETF_df_task_2 = ETF_df_task_1[['ETF 종목명','구성종목 표준코드','구성종목 종목명','편입비율']]

In [13]:
# 구성종목 중 상위 5개 추출
ETF_df_task_3 = ETF_df_task_2.sort_values(by=['ETF 종목명','편입비율'], ascending=False)

In [14]:
# 종목별 상위 5개
ETF_df_task_4= ETF_df_task_3.groupby('ETF 종목명').head(5)

In [15]:
ETF_df_task_4[ETF_df_task_4['ETF 종목명'].str.contains('반도체')]
ETF_df_task_4[ETF_df_task_4['ETF 종목명'].str.contains('전지')]
ETF_df_task_4[ETF_df_task_4['ETF 종목명'].str.contains('자율주행')]
ETF_df_task_4[ETF_df_task_4['ETF 종목명'].str.contains('금융')]

,ETF 종목명,구성종목 표준코드,구성종목 종목명,편입비율
34483,TIGER 25-12 금융채(AA-이상),KR6205498EC7,하나카드273,4.988738
34398,TIGER 25-12 금융채(AA-이상),KR6005273F23,아이엠뱅크46-02이12A-21,4.10297
34473,TIGER 25-12 금융채(AA-이상),KR6140178EB5,케이비국민카드421-1,3.300191
34461,TIGER 25-12 금융채(AA-이상),KR6079314EA3,JB 우리캐피탈524-1(지),2.485386
34475,TIGER 25-12 금융채(AA-이상),KR6145763DC9,BNK캐피탈338-3,2.48222
7606,TIGER 200 금융,KR7316140003,우리금융지주,7.825771
7587,TIGER 200 금융,KR7000810002,삼성화재,7.048532
7595,TIGER 200 금융,KR7032830002,삼성생명,5.727802
7607,TIGER 200 금융,KR7323410001,카카오뱅크,5.180046
7602,TIGER 200 금융,KR7138040001,메리츠금융지주,4.741424


## 3. 고객 데이터 시나리오
 - 고객 데이터 생성

In [16]:
Customer_A = ETF_df_task_4[ETF_df_task_4['ETF 종목명'].isin(['ACE AI반도체포커스','ACE 2차전지&친환경차액티브','KODEX 자율주행액티브','RISE 200금융'])]

In [17]:
Customer_A = Customer_A.reset_index().drop('index',axis = 1)

In [18]:
Customer_A 


,ETF 종목명,구성종목 표준코드,구성종목 종목명,편입비율
0,RISE 200금융,KR7316140003,우리금융지주,7.848761
1,RISE 200금융,KR7000810002,삼성화재,7.198696
2,RISE 200금융,KR7032830002,삼성생명,5.759096
3,RISE 200금융,KR7323410001,카카오뱅크,5.192032
4,RISE 200금융,KR7138040001,메리츠금융지주,4.756095
5,KODEX 자율주행액티브,KR7012330007,현대모비스,8.590932
6,KODEX 자율주행액티브,KR7307950006,현대오토에버,7.626326
7,KODEX 자율주행액티브,KR7000660001,SK하이닉스,5.967903
8,KODEX 자율주행액티브,KR7086280005,현대글로비스,4.925796
9,KODEX 자율주행액티브,KR7005380001,현대차,4.391058


In [19]:
Customer_A.to_csv('customer_set.csv',encoding='utf-8-sig')

In [20]:
Custer_Having_ticker_lst = list(Customer_A['구성종목 종목명'].unique())

In [21]:
Custer_Having_ticker_lst

['우리금융지주',
 '삼성화재',
 '삼성생명',
 '카카오뱅크',
 '메리츠금융지주',
 '현대모비스',
 '현대오토에버',
 'SK하이닉스',
 '현대글로비스',
 '현대차',
 '삼성전자',
 '한미반도체',
 '파크시스템스',
 'DB하이텍',
 '기아',
 'POSCO홀딩스',
 'LG에너지솔루션']

## 4. 고객 데이터 저장

In [15]:
con = duckdb.connect('../DB/Customer.db')

# Pandas DataFrame을 DuckDB에서 참조할 수 있도록 등록
con.register('temp_df', Customer_A)

# 테이블이 없다면 생성
con.execute("""
    CREATE TABLE IF NOT EXISTS Customers AS
    SELECT * FROM temp_df LIMIT 0
""")

# 데이터 삽입
con.execute("INSERT INTO Customers SELECT * FROM temp_df")

# 정리
con.unregister('temp_df')
con.close()

## 5. 고객이 보유하고 있는 데이터를 DB에 저장하기

In [48]:
import asyncio
import aiohttp
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin,urlparse
import duckdb

# ▶ 헤더
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36"
}

# ▶ 기사 리스트 파싱 함수
async def get_article_urls(session, page_url):
    try:
        async with session.get(page_url, headers=headers) as resp:
            text = await resp.text()
            soup = BeautifulSoup(text, 'html.parser')
            ul = soup.select_one('#content > div.left_cont > div > div.section.hk_news > div.section_cont > ul')
            if not ul:
                return []

            urls = []
            for a in ul.find_all('a', href=True):
                href = a['href']
                if '/article/' in href:
                    urls.append(href)
            return list(set(urls))  # 중복 제거
    except Exception as e:
        print(f"[get_article_urls error] {page_url} - {e}")
        return []

# ▶ 기사 상세 파싱 함수
async def fetch_article(session, url):
    try:
        async with session.get(url, headers=headers) as resp:
            html = await resp.text()
            soup = BeautifulSoup(html, 'html.parser')

            hostname = urlparse(url).hostname

            # ✅ 1. magazine.hankyung.com용 로직
            if 'magazine.hankyung.com' in hostname:
                return {
                    'header': soup.select_one('#contents h1.news-tit').text.strip() if soup.select_one('#contents h1.news-tit') else None,
                    'summary': None,
                    'content': soup.select_one('#magazineView').text.strip() if soup.select_one('#magazineView') else None,
                    'url': url,
                    'datetime': soup.select_one('#contents span.txt-num').text.strip() if soup.select_one('#contents span.txt-num') else None,
                }

            # ✅ 2. www.hankyung.com일 경우 기존 로직
            elif 'hankyung.com' in hostname:
                return {
                    'header': soup.select_one('h1.headline').text.strip() if soup.select_one('h1.headline') else None,
                    'summary': soup.select_one('div.summary').text.strip() if soup.select_one('div.summary') else None,
                    'content': soup.select_one('#articletxt').text.strip() if soup.select_one('#articletxt') else None,
                    'url': url,
                    'datetime': soup.select_one('div.datetime span.txt-date').text.strip() if soup.select_one('div.datetime span.txt-date') else None,
                }

            # ✅ 알 수 없는 도메인
            else:
                print(f"⚠️ 알 수 없는 호스트: {hostname}")
                return {'url': url, 'header': None, 'summary': None, 'content': None, 'datetime': None}

    except Exception as e:
        print(f"[fetch_article error] {url} - {e}")
        return {'url': url, 'header': None, 'summary': None, 'content': None, 'datetime': None}
# ▶ 메인 비동기 루프
async def extract_news_data_async(query_text, page_range):
    base_url = 'https://search.hankyung.com/search/news?query={query}&page={page}'
    search_urls = [base_url.format(query=query_text, page=p+1) for p in range(page_range)]

    async with aiohttp.ClientSession() as session:
        # 1. 페이지별 기사 링크 수집
        tasks = [get_article_urls(session, url) for url in search_urls]
        results = await asyncio.gather(*tasks)
        article_urls = list(set([url for sublist in results for url in sublist]))

        print(f"🔗 총 {len(article_urls)}개의 기사 URL 수집됨")

        # 2. 기사 본문 수집
        article_tasks = [fetch_article(session, url) for url in article_urls]
        articles = await asyncio.gather(*article_tasks)

        # 3. ticker 컬럼 추가
        for article in articles:
            article['ticker'] = query_text

        # 4. 비어 있으면 dummy row 추가
        if not articles:
            articles = [{
                'header': None,
                'summary': None,
                'content': None,
                'url': None,
                'datetime': None,
                'ticker': query_text
            }]
            print("⚠️ 수집된 기사가 없어 None 값으로 대체 저장합니다.")

        # 4. DuckDB 저장
        df = pd.DataFrame(articles)
        con = duckdb.connect('../DB/Customer_news_sim.db')

        # Pandas DataFrame을 DuckDB에서 참조할 수 있도록 등록
        con.register('temp_df', df)

        # 테이블이 없다면 생성
        con.execute("""
            CREATE TABLE IF NOT EXISTS articles AS
            SELECT * FROM temp_df LIMIT 0
        """)

        # 데이터 삽입
        con.execute("INSERT INTO articles SELECT * FROM temp_df")

        # 정리
        con.unregister('temp_df')
        con.close()

        print(f"✅ 저장 완료: ../DB/Customer_news.db (ticker = {query_text})")

# ▶ 실행 함수
def extract_news_data(query_text, page_range):
    loop = asyncio.get_event_loop()
    loop.run_until_complete(extract_news_data_async(query_text, page_range))

In [109]:
for ticker in Custer_Having_ticker_lst:
    extract_news_data(ticker,20)

🔗 총 200개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 우리금융지주)
🔗 총 200개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 삼성화재)
🔗 총 200개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 삼성생명)
🔗 총 200개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 카카오뱅크)
🔗 총 200개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 메리츠금융지주)
🔗 총 200개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 현대모비스)
🔗 총 200개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 현대오토에버)
🔗 총 200개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = SK하이닉스)
🔗 총 200개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 현대글로비스)
🔗 총 200개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 현대차)
🔗 총 200개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 삼성전자)
🔗 총 200개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 한미반도체)
🔗 총 200개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = 파크시스템스)
🔗 총 200개의 기사 URL 수집됨
✅ 저장 완료: ../DB/Customer_news.db (ticker = DB하이텍)
🔗 총 200개의 기사 URL 수

저장이 잘 되었는 지 확인

In [45]:
cus_news = duckdb.connect('../DB/Customer_news_sim.db')

In [46]:
cusA_news_df = cus_news.execute('select * from articles').fetch_df()
cus_news.close()

In [49]:
cusA_news_df.sample(200,random_state=42)

,header,summary,content,url,datetime,ticker
1024,금융권 수장들 줄줄이 임기만료…'역대급 인사 큰장' 선다,지주 회장부터 행장까지\n'인사 태풍' 몰아치나\n\n들썩이는 금융 공공기관 \n산...,올 하반기 금융권 수장 인사와 관련해 ‘역대급 큰 장’이 설 것으로 전망된다. 일부...,https://www.hankyung.com/article/2025072126601,2025.07.21 17:53,우리금융지주
2177,"나우로보틱스, 노란봉투법 통과 후 로봇주 강세 속 18% 급등",None,나우로보틱스 주가가 2025년 8월 28일 코스닥 시장에서 전 거래일 대비 18% ...,https://www.hankyung.com/article/202508287036a,2025.08.28 13:13,현대오토에버
1444,휴머노이드 ‘부활’ 신호탄...정부 정책·노란봉투법 수혜 기대에 로봇주 급등,None,보합세를 이어가던 로봇주가 정부의 경제성장전략 발표와 ‘노란봉투법’ 시행 기대감을 ...,https://www.hankyung.com/article/202508274514a,2025.08.28 10:10,삼성생명
3386,"01일, 코스닥 기관 순매수상위에 제약 업종 8종목",None,"기관 투자자는 01일 코스닥에서 알테오젠, 파마리서치, 휴젤 등을 중점적으로 사들인...",https://www.hankyung.com/article/202504014081L,2025.04.01 18:35,파크시스템스
3203,"'페이스메이커' 자처한 李…대미 투자펀드, 큰 틀서 합의",None,영상 모듈 닫기\n\n\n\n\n[앵커]우리시간으로 오늘 새벽 백악관에서 이재명 대...,https://www.hankyung.com/article/2025082629065,2025.08.26 14:52,한미반도체
...,...,...,...,...,...,...
1041,"우리금융그룹, 美 관세 피해 수출업체에 10조2000억 투입…상생대출·금리 우대",자영업 지원 '우리동네 선한가게'\n장애인 위한 물품 기부함 설치,임종룡 우리금융그룹 회장(왼쪽)과 고객이 지난 3월 굿윌기부함에 1호 기부를 하는 ...,https://www.hankyung.com/article/2025071611511,2025.07.16 15:41,우리금융지주
314,"화우, 'M&A 스타' 이진국·윤소연 변호사 영입","윤희웅 대표, 류명현 외국변호사 이어\n기업자문 분야 대거 보강","(왼쪽부터) 이진국, 윤소연 화우 변호사. 사진=화우 제공\n\n ...",https://www.hankyung.com/article/202508049948i,2025.08.04 11:43,현대오토에버
490,토큰증권(STO) 국정과제 포함에 관련주 강세…정책·외국인 수급 동반 호재,None,토큰증권(STO)이 새 정부의 5개년 국정과제에 포함되며 관련 종목들이 연일 상승세...,https://www.hankyung.com/article/202508287466a,2025.08.29 11:00,현대차
3953,"주식 신용대출 이자 부담, 무빙스탁 3%대 금리로 한숨 돌린다",None,"전송종목 : 유비케어, 엠케이전자, 유나이티드제약, 동성화인텍, SJG세종최근 주식...",https://www.hankyung.com/article/202508299043a,2025.08.29 10:37,POSCO홀딩스


In [44]:
cusA_news_df.to_csv('news.csv',encoding='utf-8-sig')

In [159]:
cusA_news_df.head(5)

,header,summary,content,url,datetime,ticker
0,"iM뱅크(아이엠뱅크), 2025 을지연습 실시",☞ 8월18일 ~ 20일 국가적 차원의 위기 대응훈련 동참\n\n☞ 위기관리 및 대...,"iM뱅크(아이엠뱅크, 은행장 황병우)는 안보 태세를 강화하고 비상 상황에서 국민의 ...",https://www.hankyung.com/article/202508201792P,2025.08.20 17:58,우리금융지주
1,2025 밸류업 CEO 50 랭킹 - 금융 11~20위,None,[커버스토리] 2025 밸류업 CEO 50\n\n\n\n\n\n\n\n\n\n\n\...,https://magazine.hankyung.com/money/article/20...,2025.08.04 10:53,우리금융지주
2,"""삼성전자보다 더 받아요""...은행원 연봉 보니 '헉'",None,사진=뉴스1국내 주요 시중은행 직원들의 올 상반기 평균 급여가 6350만원에 달하는...,https://magazine.hankyung.com/business/article...,2025.08.16 18:00,우리금융지주
3,"HD현대미포, 조선 업황 회복에 수익성 개선 기대와 수급 흐름 맞물려 상승세",None,조선업 전반의 회복세가 부각되면서 HD현대미포에 대한 투자자들의 관심이 다시 고조되...,https://www.hankyung.com/article/202508185277a,2025.08.18 13:03,우리금융지주
4,"국민이 이끄는 대한민국 경제, 이제 국민이 주주!",농협금융「우리가 주인! 내 주식 갖기」캠페인 전개 -,NH농협금융지주(회장 이찬우)는 8월 15일부터 10월 31일까지「우리가 주인! 내...,https://www.hankyung.com/article/202508125826P,2025.08.12 13:51,우리금융지주


In [160]:
cusA_news_df.shape

(4250, 6)

In [161]:
# 중복제거 필요...
cusA_news_df.groupby('ticker').count()

,header,summary,content,url,datetime
ticker,,,,,
DB하이텍,250,29,250,250,250
LG에너지솔루션,250,54,250,250,250
POSCO홀딩스,250,12,250,250,250
SK하이닉스,250,64,250,250,250
기아,250,115,250,250,250
메리츠금융지주,250,59,250,250,250
삼성생명,250,91,250,250,250
삼성전자,250,81,250,250,250
삼성화재,250,105,250,250,250


In [162]:
cusA_news_df = cusA_news_df.drop_duplicates()

In [163]:
cusA_news_df.groupby('ticker').count()

,header,summary,content,url,datetime
ticker,,,,,
DB하이텍,200,23,200,200,200
LG에너지솔루션,200,43,200,200,200
POSCO홀딩스,200,12,200,200,200
SK하이닉스,200,52,200,200,200
기아,200,93,200,200,200
메리츠금융지주,200,47,200,200,200
삼성생명,200,72,200,200,200
삼성전자,200,64,200,200,200
삼성화재,200,81,200,200,200


## 번외) 날짜 전처리 확인 : 최근 N일치 가져오는 로직 생성

In [164]:
cusA_news_df['Date'] = cusA_news_df.datetime.str[:10]
cusA_news_df = cusA_news_df.drop('datetime',axis=1)
cusA_news_df['Date'] = pd.to_datetime(cusA_news_df['Date'])

In [165]:
# 2. 기준일 계산 (오늘 날짜 - 5일)
today = pd.to_datetime("2025-08-29")
five_days_ago = today - timedelta(days=5)

# 3. 최근 5일치 필터링
recent_news_df = cusA_news_df[cusA_news_df['Date'] >= five_days_ago]

In [166]:
recent_news_df

,header,summary,content,url,ticker,Date
11,한국 디지털자산 산업의 미래는 어떤 모습인가 [태평양의 미래금융],韓 구체적 규제안 전무...예측가능성 0\n美 백악관은 디지털 자산 규제 '166페...,한경 로앤비즈의 ‘Law Street’ 칼럼은 기업과 개인에게 실용적인 법률 지식을...,https://www.hankyung.com/article/202508263699i,우리금융지주,2025-08-27
12,"iM금융그룹, UNGC-UNFCCC-UNEP 공동 ‘기후 리더십 조찬간담회’ 참여",‘기후행동 통한 비즈니스 리더십 기획 확보’ 주제 논의,iM금융그룹(회장 황병우)은 지난 27일 서울 그랜드하얏트 호텔에서 열린 UNGC(...,https://www.hankyung.com/article/202508287972P,우리금융지주,2025-08-28
14,"“I LOVE iM!”…iM금융그룹, 건강한 기업문화 조성 박차",‘우리 회사 사랑하기’ 캠페인 진행\n\n계열사 모바일 서비스 이용 장려,iM금융그룹(회장 황병우)은 오는 9월 말까지 전 계열사 임직원을 대상으로 「I L...,https://www.hankyung.com/article/202508262387P,우리금융지주,2025-08-26
17,"휴먼시아 청약, 다시 생각해 볼게요 [김용우의 각개전투]",공공주택도 대형 건설사 브랜드 가능\n프리미엄 이미지·입찰 허들에 '멈칫'\n보급형...,한경 로앤비즈의 'Law Street' 칼럼은 기업과 개인에게 실용적인 법률 지식을...,https://www.hankyung.com/article/202508237248i,우리금융지주,2025-08-26
18,"증권주, 美 금리인하 기대감에 동반 강세",None,제롬 파월 미 중앙은행(Fed) 의장이 지난 22일(현지시간) 미 와이오밍주 잭슨홀...,https://www.hankyung.com/article/2025082591866,우리금융지주,2025-08-25
...,...,...,...,...,...,...
4232,증권사 신용융자 쓰시는 분? 이 상품 하나면 3%대 금리로 가능합니다,None,"전송종목 : HD현대중공업, 두산퓨얼셀, 하이브, 솔브레인, SK아이이테크놀로지최근...",https://www.hankyung.com/article/202508258834a,LG에너지솔루션,2025-08-25
4237,이재용 회장 방미사절단 출국…삼성·SK·LG 등 재계 총출동,None,이재용 삼성전자 회장이 24일 오후 서울 강서구 김포비즈니스항공센터에서 한미 정상회...,https://www.hankyung.com/article/2025082482927,LG에너지솔루션,2025-08-24
4241,"증권사 신용 이용자들 주목! 3%대 금리의 상품 출시!!, 매도없이 전환가능",None,"전송종목 : LG에너지솔루션, DL이앤씨, F&F, SK스퀘어, 에코프로머티최근 주...",https://www.hankyung.com/article/202508258835a,LG에너지솔루션,2025-08-25
4244,[단독] 5조 갚아야하는데 현금 '텅텅'…줄줄이 '초비상',10대 석화기업 단기 차입금만 5.3兆 \n대주주 증자 압박 커진다\n\n1년내 만...,"국내 10대 석유화학 기업이 발행한 회사채, 기업어음(CP) 가운데 1년 안에 만기...",https://www.hankyung.com/article/2025082484091,LG에너지솔루션,2025-08-24


In [167]:
recent_news_df= recent_news_df.drop_duplicates()

In [168]:
# 최근 30일치 뉴스 기사 필터링 시 남아있는 뉴스 기사 개수
recent_news_df.groupby('ticker').count()

,header,summary,content,url,Date
ticker,,,,,
DB하이텍,2,0,2,2,2
LG에너지솔루션,75,15,75,75,75
POSCO홀딩스,51,0,51,51,51
SK하이닉스,132,36,132,132,132
기아,62,30,62,62,62
메리츠금융지주,5,2,5,5,5
삼성생명,17,7,17,17,17
삼성전자,200,64,200,200,200
삼성화재,19,9,19,19,19


## 정규식으로 뉴스 기사 내 불용어 처리

In [169]:
boilerplate_patterns = [
    r"\* 아래 텍스트는 실제 방송 내용과 차이가 있을 수 있으니.*",
    r"\*인터뷰를 인용보도할 때는 프로그램명.*",
    r"저작권은.*에 있습니다.*",
    r"▶ 알립니다.*",
    r"\[앵커\].*?\[",
    r"\[기자\].*?\[",
    r"\[.*?\]",             # 모든 대괄호 안 내용
    r"영상취재:.*",
    r"영상편집:.*",
    r"그래픽:.*",
    r"\/?사진=(연합뉴스|뉴스\S*)[^\n]*\s*"       # 사진=출처 or /사진=출처 패턴 제거
]

In [170]:
compiled_pattern = re.compile("|".join(boilerplate_patterns))

In [171]:
test_pattern = cusA_news_df.content.astype(str).apply(lambda x : compiled_pattern.sub("", x).strip())

In [172]:
cusA_news_df.content.astype(str).apply(lambda x : len(x)).describe()

count      3400.000000
mean       1689.115882
std        6914.147389
min           0.000000
25%         726.000000
50%        1080.000000
75%        1500.000000
max      125274.000000
Name: content, dtype: float64

In [173]:
test_pattern.apply(lambda x : len(x)).describe()

count      3400.000000
mean       1673.633529
std        6912.770281
min           0.000000
25%         716.000000
50%        1077.000000
75%        1499.250000
max      125274.000000
Name: content, dtype: float64

#### 결론 : 큰 영향 없다..? 그냥 해보자

## 질문 생성 (For Labeling)  -- 주말 작업 예정 .. 모델 성능평가를 위함
Part_1 : Re-Ranking 라벨링

In [174]:
def labeling(data,ticker):
  template = """
  <instruction>
  다음은 뉴스 기사 본문과 해당 기사에 매핑된 종목명(티커)입니다.
  당신의 임무는 기사 본문이 해당 종목에 대한 기사인지 여부를 판별하는 것입니다.

  규칙:
  - 1 : 뉴스 내용이 해당 종목의 시황을 나타낼 경우.
     예: 종목명이 기사에 등장하고, 그 종목의 실적, 주가, 제품, 사건, 경영, 산업 동향 등과 밀접한 관련이 있음.
     예) 종목의 실적/주가/사업/이슈/계약/정책/규제/소송/리스크/전망 등.
     예) 섹터 기사라도 해당 종목이 사례/주요 구성원으로 명시적 언급되고 맥락에 기여.
  - 0 : 뉴스 내용이 해당 종목을 단순 언급하는 경우이거나, 직접적으로 관련이 없음  
     예: 종목명이 전혀 등장하지 않거나, 비슷한 용어를 가진 단어의 내용이 등장하더라도 다른 주제가 메인인 경우.
     예) 피상적 나열(태그/키워드/꼬리말 광고)만 존재, 타사 이슈가 중심.
     
  출력 형식:
  - 숫자 1 또는 0만 출력
  </instruction>

  예시:
  ---
  [티커] 삼성전자
  [본문] 삼성전자가 2분기 실적 호조를 발표하며 주가가 3% 상승했다.
  [정답] 1
  ---
  [티커] 삼성전자
  [본문] 미국 증시가 기술주 중심으로 상승세를 보였다. 애플과 구글 주가가 상승했다.
  [정답] 0
  ---

  다음 데이터를 분류하세요.

  [티커] {ticker_name}
  [본문] {news_content}
  [정답]
  """

  prompt = ChatPromptTemplate.from_template(template)

  # 3. 체인 구성
  chain = prompt | llm | StrOutputParser()

  # 4. 실행 예시
  query = chain.invoke({"news_content": data,'ticker_name' : ticker})
  return query

## 6. VectorDB 저장

In [4]:
# 4. OpenAI 임베딩 모델 로딩
embedding = OpenAIEmbeddings(model="text-embedding-3-large")

persist_directory = "../VectorDB/chroma_news_db"

vectordb = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding,
    collection_name="SLM_News_5") 

In [176]:
def pick_splitter_by_length(text_len: int) -> RecursiveCharacterTextSplitter:
    """
    뉴스 본문의 길이에 따라 적절한 텍스트 분할기를 반환합니다.
    """
    if text_len <= 1200:
        # 짧은 기사 → 굳이 자르지 않고 1덩어리로 처리
        return RecursiveCharacterTextSplitter(
            chunk_size=1200,
            chunk_overlap=0,
            separators=["\n\n", "\n", " ", ""]
        )
    elif text_len <= 10_000:
        # 중간 길이 → 일반적인 1,200자 기준으로 분할
        return RecursiveCharacterTextSplitter(
            chunk_size=1200,
            chunk_overlap=150,
            separators=["\n\n", "\n", " ", ""]
        )
    elif text_len <= 50_000:
        # 긴 기사 → 덩어리를 좀 더 키움
        return RecursiveCharacterTextSplitter(
            chunk_size=1800,
            chunk_overlap=200,
            separators=["\n\n", "\n", " ", ""]
        )
    else:
        # 초장문 → 더 크게 자르되, 요약도 고려 (이건 후속 처리 필요)
        return RecursiveCharacterTextSplitter(
            chunk_size=2000,
            chunk_overlap=200,
            separators=["\n\n", "\n", " ", ""]
        )
    

In [177]:
# 2. 문서 리스트 생성 (chunk + metadata 포함)
def make_documents(df):
    docs = []

    for idx, row in tqdm(df.iterrows()):
        text = row["content"]
        splitter = pick_splitter_by_length(len(text))
        chunks = splitter.split_text(text)
        for i, chunk in enumerate(chunks):
            label = labeling(chunks,row['ticker'])
            metadata = {
                "title": row["header"],
                "url": row["url"],
                "Date": row["Date"],
                "ticker": row.get("ticker", "None"),
                "chunk_idx": i,
                "original_idx": idx,
                'label' : label  #labeling한 결과를 넣자!! 
            }
            time.sleep(0.1)
            docs.append(Document(page_content=chunk, metadata=metadata))

    return docs


In [178]:
def get_recent_articles(df: pd.DataFrame, ticker: str, days: int = 5):
    df['Date'] = pd.to_datetime(df['Date'])
    today = df['Date'].max() # 가지고 있는 뉴스의 가장 최신 데이터
    recent_df = df[
        (df['ticker'] == ticker) &
        (df['Date'] >= today - timedelta(days=days))
    ]
    return recent_df.sort_values(by="Date", ascending=False)


In [179]:
def make_doc_id(d: Document) -> str:
    """
    url + chunk_idx(없으면 0) + 시간
    """
    run_utc = datetime.now(timezone.utc)
    base = f"{d.metadata.get('url','')}_{d.metadata.get('chunk_idx', 0)}_{run_utc}"
    return md5(base.encode("utf-8")).hexdigest()

def chunks(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]


## Label작업 수행

In [136]:
for ticker in Custer_Having_ticker_lst:
    print(f"========= ticker {ticker} 진행 중~ =============")
    task_1_df = get_recent_articles(cusA_news_df,ticker=ticker,days = 20)
    task_1_df['Date'] = task_1_df.Date.astype('str')
    # document 만들기
    task_1_docs = make_documents(task_1_df)
    length_docs = len(task_1_docs)
    print(f'각 {ticker} 별 docs의 개수 : {length_docs}')
    BATCH = 32  # 상황에 맞게 조절

    for docs in tqdm(chunks(task_1_docs, BATCH), total=(len(task_1_docs) + BATCH - 1) // BATCH):

        ids = [make_doc_id(d) for d in docs]
        vectordb.add_documents(documents=docs, ids=ids)
        time.sleep(0.1) 

========= ticker 우리금융지주 진행 중~ =============


33it [00:45,  1.39s/it]


각 우리금융지주 별 docs의 개수 : 58


100%|██████████| 2/2 [00:03<00:00,  1.83s/it]


========= ticker 삼성화재 진행 중~ =============


...

100%|██████████| 8/8 [00:19<00:00,  2.42s/it]


========= ticker LG에너지솔루션 진행 중~ =============


195it [04:30,  1.39s/it]


각 LG에너지솔루션 별 docs의 개수 : 379


100%|██████████| 12/12 [00:25<00:00,  2.16s/it]


In [181]:
print("Number of documents in DB:", vectordb._collection.count())

Number of documents in DB: 3129


## 7. cross-encoder 모델(w/langchain 예시)

In [87]:
import numpy as np
import pandas as pd

def _dcg_at_k(labels, k=50):
    L = np.asarray(labels)[:k]
    if L.size == 0: return 0.0
    discounts = 1.0 / np.log2(np.arange(2, L.size + 2))
    return float(np.sum(L * discounts))

def ndcg_at_k(labels, k=50):
    labels = np.asarray(labels)
    dcg  = _dcg_at_k(labels, k)
    idcg = _dcg_at_k(np.sort(labels)[::-1], k)
    return 0.0 if idcg == 0 else dcg / idcg

def precision_at_k(labels, k=50):
    L = np.asarray(labels)[:k]
    return float(L.mean()) if L.size else 0.0

def recall_at_k(labels, total_relevant, k=50):
    if not total_relevant or total_relevant <= 0: return 0.0
    return float(np.sum(np.asarray(labels)[:k])) / float(total_relevant)

def mrr_at_k(labels, k=50):
    L = np.asarray(labels)[:k]
    hit = np.where(L > 0)[0]
    return 0.0 if hit.size == 0 else 1.0 / (hit[0] + 1)

def map_at_k(labels, total_relevant, k=50):
    L = np.asarray(labels)[:k]
    if L.sum() == 0: return 0.0
    precisions, hit = [], 0
    for i, y in enumerate(L, start=1):
        if y:
            hit += 1
            precisions.append(hit / i)
    denom = max(1, min(total_relevant if total_relevant is not None else int(L.sum()), k))
    return float(np.sum(precisions) / denom)

def eval_reranker_chunk(pool_df, ranked_df, k=50, rank_col="rank", score_col="relevance_score"):
    # 1) 풀: label만 필요 (score/rank 불필요)
    total_rel_pool = int(
        pd.to_numeric(pool_df["label"], errors="coerce").fillna(0).clip(0,1).sum()
    )
    print(f'total_rel_pool : {total_rel_pool}')

    # 2) 랭크드: 순서가 필요 (rank 우선, 없으면 score로 정렬, 둘 다 없으면 현재 순서 사용)
    g = ranked_df.copy()
    if rank_col in g.columns:
        g = g.sort_values(rank_col, ascending=True)
    elif score_col in g.columns:
        g = g.sort_values(score_col, ascending=False)
        g[rank_col] = np.arange(1, len(g)+1)
    else:
        g[rank_col] = np.arange(1, len(g)+1)

    y_topk = pd.to_numeric(g["label"], errors="coerce").fillna(0).clip(0,1).astype(int).values[:k]
    print(f'y_topk : {y_topk}')
    return {
        f"Precision@{k}": precision_at_k(y_topk, k),
        f"Recall@{k}(pool)": recall_at_k(y_topk, total_rel_pool, k),
        f"MRR@{k}": mrr_at_k(y_topk, k),
        f"MAP@{k}": map_at_k(y_topk, total_rel_pool, k),
        f"nDCG@{k}": ndcg_at_k(y_topk, k),
        "PoolSize": int(len(pool_df)),
        "PoolRelevant": total_rel_pool,
        "TopK": int(min(len(g), k)),
        "TopKRelevant": int(y_topk.sum()),
    }


Re-ranker 사용하기 전

In [63]:
retriver_target = ['우리금융지주','삼성화재','카카오뱅크','삼성생명','메리츠금융지주','현대모비스','현대글로비스','SK하이닉스','현대오토에버']

In [64]:
retriever_total = vectordb.as_retriever(search_kwargs={"k": 50,'filter' : {'ticker' : retriver_target}})

In [65]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i + 1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

In [78]:
before_lst = []

In [79]:
for ticker in retriver_target:
    retriever = vectordb.as_retriever(search_kwargs={"k": 50,'filter' : {'ticker' : ticker}})
    CrossEncoder_prompt = f'''
    이 뉴스들 중에서 "{ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, {ticker}가 단순히 함께 언급된 기사라면 제외해줘.
    '''
    print(CrossEncoder_prompt.strip())
    raw_docs = retriever.get_relevant_documents(CrossEncoder_prompt)
    before_lst.extend(raw_docs)
    print(list(map(lambda x: x.metadata['label'],raw_docs))[:5])
    print(list(map(lambda x: x.page_content,raw_docs))[:5])

이 뉴스들 중에서 "우리금융지주"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, 우리금융지주가 단순히 함께 언급된 기사라면 제외해줘.
['1', '1', '1', '1', '1']
['<앵커> 오늘 금융지주 주가가 오전에 잠시 조정받는 듯했는데 KB금융 제외하곤 낙폭을 줄여가는 모습입니다. 앞서 최민정 기자 언급한 자사주 의무 소각 관련해 금융지주들에는 어떤 영향이 있을지 경제부 유주안 기자와 이야기 나눠봅니다. 금융지주들은 자사주 매입과 소각에 꽤 적극적인데, 보유중인 자사주의 비중은 어떻습니까?<기자>4대 금융지주 자사주 보유 현황(KB 4.54%, 신한 2.09%, 하나 3.77%, 우리 1.16%)은, 보시다시피 비중이 크다고 보이진 않습니다.자사주 소각 의무화 추진에 따른 수혜주로 꼽히는 미래에셋(22.98) 대신(25.12) 신영(52.58%) 등에 비하면 매우 작은 수준이고요, 일반지주사들(티와이홀딩스 29.79%, SK 24.8%, 롯데지주 27.37% 등)에 비해서도 비중이 낮습니다.과거 역사를 살펴보면 금융주에 있어서는 자사주 ‘지배의 도구’보다는 인수합병 때 주식교환을 하기 위해, 전략적 투자자와 주식을 스왑하는 차원에서 일정 부분 보유해왔던 것을 알 수 있고요, 최근 들어서는 매입하면 바로 소각해서 주식의 가치를 올리는, 주주환원의 수단으로 자사주를 활용해 왔습니다.따라서, 자사주 소각 의무화에 따른 강제성 측면에서 금융지주에 미치는 단기적 직접적 영향은 크지 않겠습니다. 중장기적으로 볼 때 금융지주의 주주환원에 보다 힘이 실릴 것이란 기대감으로 이어질 수 있는데요,정책이 추구하는 목표를 이미 금융지주사들이 실행해 옮기고 있는 것이고, 향후 상장사들 사이에 자사주 매입소각이 보편화되면 재무적 여력과, 주주환원에 대한 강한 의지를 가진 금융사들이 더 적극적인 주주환원을 펼칠 수 있는 배경이 될 수 있을 것입니다.또한, 정부가 함께 추진중인 배당소득 분리과세 정책은 배당성향이 높고, 분기배당에 적극적인 금융주에게 특히

In [80]:
before_lst[0]

Document(id='31e5a84962c84ec540a883b882579f4c', metadata={'title': '더 커진 주주환원 기대감…금융주, 어닝시즌 관전포인트', 'original_idx': 245, 'chunk_idx': 1, 'label': '1', 'ticker': '우리금융지주', 'Date': '2025-07-09', 'url': 'https://www.hankyung.com/article/2025070981175'}, page_content='<앵커> 오늘 금융지주 주가가 오전에 잠시 조정받는 듯했는데 KB금융 제외하곤 낙폭을 줄여가는 모습입니다. 앞서 최민정 기자 언급한 자사주 의무 소각 관련해 금융지주들에는 어떤 영향이 있을지 경제부 유주안 기자와 이야기 나눠봅니다. 금융지주들은 자사주 매입과 소각에 꽤 적극적인데, 보유중인 자사주의 비중은 어떻습니까?<기자>4대 금융지주 자사주 보유 현황(KB 4.54%, 신한 2.09%, 하나 3.77%, 우리 1.16%)은, 보시다시피 비중이 크다고 보이진 않습니다.자사주 소각 의무화 추진에 따른 수혜주로 꼽히는 미래에셋(22.98) 대신(25.12) 신영(52.58%) 등에 비하면 매우 작은 수준이고요, 일반지주사들(티와이홀딩스 29.79%, SK 24.8%, 롯데지주 27.37% 등)에 비해서도 비중이 낮습니다.과거 역사를 살펴보면 금융주에 있어서는 자사주 ‘지배의 도구’보다는 인수합병 때 주식교환을 하기 위해, 전략적 투자자와 주식을 스왑하는 차원에서 일정 부분 보유해왔던 것을 알 수 있고요, 최근 들어서는 매입하면 바로 소각해서 주식의 가치를 올리는, 주주환원의 수단으로 자사주를 활용해 왔습니다.따라서, 자사주 소각 의무화에 따른 강제성 측면에서 금융지주에 미치는 단기적 직접적 영향은 크지 않겠습니다. 중장기적으로 볼 때 금융지주의 주주환원에 보다 힘이 실릴 것이란 기대감으로 이어질 수 있는데요,정책이 추구하는 목표를 이미 금융지주사들이 실행해 옮기고 있는 것이고, 향후 상장사들 사이

In [81]:
rerank_df_before = pd.DataFrame(list(map(lambda x: x.metadata,before_lst)))

In [82]:
rerank_df_before

,title,original_idx,chunk_idx,label,ticker,Date,url
0,"더 커진 주주환원 기대감…금융주, 어닝시즌 관전포인트",245,1,1,우리금융지주,2025-07-09,https://www.hankyung.com/article/2025070981175
1,대출 수익성 악화에…4대 금융 실적 꺾였다,158,0,1,우리금융지주,2025-07-15,https://www.hankyung.com/article/2025071501731
2,4대금융 2분기 순익 5.4조…사상 최대 실적,475,0,1,우리금융지주,2025-07-25,https://www.hankyung.com/article/2025072528861
3,"'우리금융지주' 52주 신고가 경신, 전일 외국인 대량 순매수",237,0,1,우리금융지주,2025-07-14,https://www.hankyung.com/article/202507145874L
4,"더 커진 주주환원 기대감…금융주, 어닝시즌 관전포인트",245,3,1,우리금융지주,2025-07-09,https://www.hankyung.com/article/2025070981175
...,...,...,...,...,...,...,...
417,[인천테크노파크 2025년 인천 라이징스타 선정기업] 실시간 항공난류 진단 및 예측...,3412,2,1,현대오토에버,2025-08-07,https://magazine.hankyung.com/job-joy/article/...
418,[인천테크노파크 2025년 인천 라이징스타 선정기업] 실시간 항공난류 진단 및 예측...,3412,3,1,현대오토에버,2025-08-07,https://magazine.hankyung.com/job-joy/article/...
419,"화우, 'M&A 스타' 이진국·윤소연 변호사 영입",3063,0,0,현대오토에버,2025-08-04,https://www.hankyung.com/article/202508049948i
420,[인천테크노파크 2025년 인천 라이징스타 선정기업] 실시간 항공난류 진단 및 예측...,3412,0,1,현대오토에버,2025-08-07,https://magazine.hankyung.com/job-joy/article/...


In [85]:
VecDB = vectordb._collection
total_results = VecDB.get(
         include = ['documents','metadatas']   
        )

In [86]:
total_df = pd.DataFrame(total_results['metadatas'])

In [106]:
total_df

,original_idx,Date,title,ticker,label,url,chunk_idx
0,236,2025-08-07,금융사 채권 발행 봇물…저금리에 자본 확충,우리금융지주,1,https://www.hankyung.com/article/2025080765681,0
1,236,2025-08-07,금융사 채권 발행 봇물…저금리에 자본 확충,우리금융지주,1,https://www.hankyung.com/article/2025080765681,1
2,236,2025-08-07,금융사 채권 발행 봇물…저금리에 자본 확충,우리금융지주,1,https://www.hankyung.com/article/2025080765681,2
3,413,2025-08-07,"07일, 거래소 외국인 순매수상위에 전기,전자 업종 4종목",우리금융지주,1,https://www.hankyung.com/article/202508078176L,0
4,209,2025-08-06,"[마켓PRO] Today's Pick : ""에코프로비엠, 3분기 흑자전환 기대""",우리금융지주,1,https://www.hankyung.com/article/202508063668i,0
...,...,...,...,...,...,...,...
1899,4130,2025-07-10,"현대글로비스, '대·중소기업 안전 상생' 고용부 장관상",현대글로비스,1,https://www.hankyung.com/article/202507109861i,0
1900,4485,2025-07-10,"현대글로비스, 협력사 안전 상생 최우수기업으로 고용부장관상 수상",현대글로비스,1,https://www.hankyung.com/article/202507109010P,0
1901,4080,2025-07-09,美 관세에 웃는 '현대차 삼형제',현대글로비스,1,https://www.hankyung.com/article/2025070982441,0
1902,4260,2025-07-08,9~10% 주식 관련 대출에서 3%대 대출로 교체,현대글로비스,1,https://www.hankyung.com/article/202507083739a,0


In [88]:
rerank_eval = []

In [89]:
for kind in retriver_target:
    total_df_target = total_df[total_df.ticker==kind]
    rerank_df_target = rerank_df_before[rerank_df_before.ticker==kind]
    rerank_eval.append(eval_reranker_chunk(total_df_target,rerank_df_target))

total_rel_pool : 191
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 94
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 0 1 1 1 0 1 1 0 1 1 1 1 1 1 1 1 0 1 1 1 1 1
 1 1 1 1 1 0 1 1 1 1 0 0 1]
total_rel_pool : 119
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 110
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 0 1 1 1 0 0 0 1 1 1 1]
total_rel_pool : 49
y_topk : [1 1 1 1 1 1 0 1 1 1 1 1 1 0 0 1 1 0 0 1 1 1 1 1 0 1 1 1 1 1 0 1 1 0 1 1 0
 1 1 1 1 1 0 0 0 0 1 0 1 1]
total_rel_pool : 122
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 60
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 949
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1

In [90]:
rerank_eval_con = dict(zip(retriver_target,rerank_eval))

In [91]:
pd.DataFrame(rerank_eval_con)

,우리금융지주,삼성화재,카카오뱅크,삼성생명,메리츠금융지주,현대모비스,현대글로비스,SK하이닉스,현대오토에버
Precision@50,1.00000,0.840000,0.980000,0.920000,0.720000,0.980000,0.960000,1.000000,0.909091
Recall@50(pool),0.26178,0.446809,0.411765,0.418182,0.734694,0.401639,0.800000,0.052687,1.000000
MRR@50,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
MAP@50,1.00000,0.762385,0.969249,0.911969,0.611900,0.969916,0.925175,1.000000,0.985283
nDCG@50,1.00000,0.978525,0.997838,0.998426,0.956682,0.997991,0.991895,1.000000,0.996850
PoolSize,219.00000,123.000000,122.000000,193.000000,66.000000,128.000000,63.000000,968.000000,22.000000
PoolRelevant,191.00000,94.000000,119.000000,110.000000,49.000000,122.000000,60.000000,949.000000,20.000000
TopK,50.00000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,22.000000
TopKRelevant,50.00000,42.000000,49.000000,46.000000,36.000000,49.000000,48.000000,50.000000,20.000000


Re-ranker 모델 사용 후

In [92]:
#model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
compressor = CrossEncoderReranker(model=model, top_n=50)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever_total
)

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


검색 유사도 측정
 - 이유 : 적정한 조건을 찾기 위함

In [93]:
scored_docs = []

retriever가 내부에서 메타데이터 필터(where/filter) 를 쓰고 있는데, 거기에 ['우리금융지주', ...] 같은 리스트를 그대로 넣어둔 상태예요. 대부분의 벡터스토어(특히 Chroma/LangChain)는 where/filter 값이 단일 값(str/int/float) 이거나 연산자 표현식이어야 하고, 리스트는 $in 같은 연산자로 감싸줘야 합니다. 그래서 get_relevant_documents() 부를 때마다 같은 잘못된 필터가 적용되어 터진 거죠.

빠른 해결책 (권장)
루프마다 단일 종목으로 필터를 바꿔서 조회하세요.

In [94]:
for ticker in retriver_target:
    retriever = vectordb.as_retriever(search_kwargs={"k": 50,'filter' : {'ticker' : ticker}})
    CrossEncoder_prompt = f'''
    이 뉴스들 중에서 "{ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, {ticker}가 단순히 함께 언급된 기사라면 제외해줘.
    '''
    print(CrossEncoder_prompt.strip())
    raw_docs = retriever.get_relevant_documents(CrossEncoder_prompt)
    pairs = [(CrossEncoder_prompt, d.page_content) for d in raw_docs]
    scores = model.score(pairs)

    # 3) 점수 붙이고 재정렬
    for d, s in zip(raw_docs, scores):
        dd = deepcopy(d)
        dd.metadata["relevance_score"] = float(s)
        scored_docs.append(dd)
    #scored_docs.sort(key=lambda x: (x.metadata[''],x.metadata["relevance_score"]), reverse=True)

이 뉴스들 중에서 "우리금융지주"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, 우리금융지주가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


이 뉴스들 중에서 "삼성화재"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, 삼성화재가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


이 뉴스들 중에서 "카카오뱅크"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, 카카오뱅크가 단순히 함께 언급된 기사라면 제외해줘.


...

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


이 뉴스들 중에서 "SK하이닉스"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, SK하이닉스가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


이 뉴스들 중에서 "현대오토에버"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, 현대오토에버가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [95]:
# 4) 확인 - raw data
for i, d in enumerate(scored_docs, 1):
    if d.metadata['ticker'] == 'SK하이닉스':
        print(f"{i:02d} | ticker : {d.metadata['ticker']}| original_idx : {d.metadata['original_idx']} | chunk_idx : {d.metadata['chunk_idx']}| score : {d.metadata['relevance_score']:.4f} | {d.metadata.get('title')}")

351 | ticker : SK하이닉스| original_idx : 3807 | chunk_idx : 0| score : 0.0024 | SK하이닉스 "올 하반기도 D램 긍정적 업황…내년 HBM 수요 대비 선제 투자"
352 | ticker : SK하이닉스| original_idx : 3596 | chunk_idx : 0| score : 0.0721 | SK하이닉스 37만원까지 간다고?…종토방 개미들 '술렁' [선한결의 이기업 왜이래]
353 | ticker : SK하이닉스| original_idx : 3967 | chunk_idx : 0| score : 0.3013 | '꿈의 직장'서 밀린 삼성전자…1위는 '이곳'
354 | ticker : SK하이닉스| original_idx : 3541 | chunk_idx : 0| score : 0.1787 | SK하이닉스, 2분기 사상 최대 실적에 주가 3% 강세
355 | ticker : SK하이닉스| original_idx : 3922 | chunk_idx : 0| score : 0.3585 | SK하이닉스, AI 메모리 힘입어 실적·주가 동반 강세 지속
356 | ticker : SK하이닉스| original_idx : 3955 | chunk_idx : 0| score : 0.0815 | [마켓PRO] "주가조정은 기회"…고수들, SK하이닉스·한화엔진 매수
357 | ticker : SK하이닉스| original_idx : 3653 | chunk_idx : 0| score : 0.0639 | [속보] SK하이닉스, 2분기 역대 최대 매출·영업이익 달성
358 | ticker : SK하이닉스| original_idx : 3867 | chunk_idx : 1| score : 0.2453 | SK하이닉스, 삼성전자 제치고 ‘대학생이 일하고 싶은 기업’ 첫 1위
359 | ticker : SK하이닉스| original_idx : 3712 | chunk_idx : 1| score : 0.2636 | SK하이닉스, HBM 

### Re-Ranker 모델 측정

In [96]:
rerank_df = pd.DataFrame(list(map(lambda x: x.metadata,scored_docs)))

In [97]:
rerank_df.ticker.value_counts()

ticker
우리금융지주     50
삼성화재       50
카카오뱅크      50
삼성생명       50
메리츠금융지주    50
현대모비스      50
현대글로비스     50
SK하이닉스     50
현대오토에버     22
Name: count, dtype: int64

In [98]:
rerank_df.head()

,Date,label,ticker,title,chunk_idx,original_idx,url,relevance_score
0,2025-07-09,1,우리금융지주,"더 커진 주주환원 기대감…금융주, 어닝시즌 관전포인트",1,245,https://www.hankyung.com/article/2025070981175,0.113008
1,2025-07-15,1,우리금융지주,대출 수익성 악화에…4대 금융 실적 꺾였다,0,158,https://www.hankyung.com/article/2025071501731,0.010040
2,2025-07-25,1,우리금융지주,4대금융 2분기 순익 5.4조…사상 최대 실적,0,475,https://www.hankyung.com/article/2025072528861,0.215311
3,2025-07-14,1,우리금융지주,"'우리금융지주' 52주 신고가 경신, 전일 외국인 대량 순매수",0,237,https://www.hankyung.com/article/202507145874L,0.602175
4,2025-07-09,1,우리금융지주,"더 커진 주주환원 기대감…금융주, 어닝시즌 관전포인트",3,245,https://www.hankyung.com/article/2025070981175,0.004671


In [99]:
rerank_df= rerank_df.sort_values(by=['ticker','relevance_score'],ascending=False)

In [100]:
rerank_df.shape

(422, 8)

In [241]:
rerank_df[rerank_df.ticker=='SK하이닉스'].head()

,chunk_idx,title,label,original_idx,ticker,url,Date,relevance_score
377,1,SK하이닉스 사상 최대 실적에도…'신중론' 여전한 이유 [종목+],1,3927,SK하이닉스,https://www.hankyung.com/article/2025072510306,2025-07-25,0.672207
390,0,"'역대급 실적' SK하이닉스, 2분기 영업이익 9조원 넘었다",1,3724,SK하이닉스,https://magazine.hankyung.com/business/article...,2025-07-24,0.611750
366,0,"AI 수요 폭증과 HBM 기술 주도에 힘입은 SK하이닉스, 주가 반등 흐름 가속화",1,3744,SK하이닉스,https://www.hankyung.com/article/202507313103a,2025-07-31,0.601519
360,2,"SK하이닉스, 삼성전자 제치고 ‘대학생이 일하고 싶은 기업’ 첫 1위",1,3867,SK하이닉스,https://magazine.hankyung.com/job-joy/article/...,2025-07-28,0.455363
365,0,"SK하이닉스, 삼성전자 제쳤다…메모리 시장 첫 '1위'",1,3815,SK하이닉스,https://www.hankyung.com/article/2025073147077,2025-07-31,0.431711


In [179]:
rerank_df.ticker.value_counts()

ticker
SK하이닉스     50
메리츠금융지주    50
삼성생명       50
삼성화재       50
우리금융지주     50
카카오뱅크      50
현대글로비스     50
현대모비스      50
현대오토에버     22
Name: count, dtype: int64

In [180]:
rerank_df[rerank_df.ticker=='SK하이닉스'].label.value_counts()

label
1    49
0     1
Name: count, dtype: int64

In [83]:
VecDB = vectordb._collection
total_results = VecDB.get(
         include = ['documents','metadatas']   
        )

In [84]:
total_df = pd.DataFrame(total_results['metadatas'])

In [183]:
total_df.shape

(1904, 7)

In [184]:
total_df.ticker.value_counts()

ticker
SK하이닉스     968
우리금융지주     219
삼성생명       193
현대모비스      128
삼성화재       123
카카오뱅크      122
메리츠금융지주     66
현대글로비스      63
현대오토에버      22
Name: count, dtype: int64

In [185]:
total_df[total_df.ticker=='SK하이닉스'].label.value_counts()

label
1    797
0    171
Name: count, dtype: int64

In [101]:
import numpy as np
import pandas as pd

def _dcg_at_k(labels, k=50):
    L = np.asarray(labels)[:k]
    if L.size == 0: return 0.0
    discounts = 1.0 / np.log2(np.arange(2, L.size + 2))
    return float(np.sum(L * discounts))

def ndcg_at_k(labels, k=50):
    labels = np.asarray(labels)
    dcg  = _dcg_at_k(labels, k)
    idcg = _dcg_at_k(np.sort(labels)[::-1], k)
    return 0.0 if idcg == 0 else dcg / idcg

def precision_at_k(labels, k=50):
    L = np.asarray(labels)[:k]
    return float(L.mean()) if L.size else 0.0

def recall_at_k(labels, total_relevant, k=50):
    if not total_relevant or total_relevant <= 0: return 0.0
    return float(np.sum(np.asarray(labels)[:k])) / float(total_relevant)

def mrr_at_k(labels, k=50):
    L = np.asarray(labels)[:k]
    hit = np.where(L > 0)[0]
    return 0.0 if hit.size == 0 else 1.0 / (hit[0] + 1)

def map_at_k(labels, total_relevant, k=50):
    L = np.asarray(labels)[:k]
    if L.sum() == 0: return 0.0
    precisions, hit = [], 0
    for i, y in enumerate(L, start=1):
        if y:
            hit += 1
            precisions.append(hit / i)
    denom = max(1, min(total_relevant if total_relevant is not None else int(L.sum()), k))
    return float(np.sum(precisions) / denom)

def eval_reranker_chunk(pool_df, ranked_df, k=50, rank_col="rank", score_col="relevance_score"):
    # 1) 풀: label만 필요 (score/rank 불필요)
    total_rel_pool = int(
        pd.to_numeric(pool_df["label"], errors="coerce").fillna(0).clip(0,1).sum()
    )
    print(f'total_rel_pool : {total_rel_pool}')

    # 2) 랭크드: 순서가 필요 (rank 우선, 없으면 score로 정렬, 둘 다 없으면 현재 순서 사용)
    g = ranked_df.copy()
    if rank_col in g.columns:
        g = g.sort_values(rank_col, ascending=True)
    elif score_col in g.columns:
        g = g.sort_values(score_col, ascending=False)
        g[rank_col] = np.arange(1, len(g)+1)
    else:
        g[rank_col] = np.arange(1, len(g)+1)

    y_topk = pd.to_numeric(g["label"], errors="coerce").fillna(0).clip(0,1).astype(int).values[:k]
    print(f'y_topk : {y_topk}')
    return {
        f"Precision@{k}": precision_at_k(y_topk, k),
        f"Recall@{k}(pool)": recall_at_k(y_topk, total_rel_pool, k),
        f"MRR@{k}": mrr_at_k(y_topk, k),
        f"MAP@{k}": map_at_k(y_topk, total_rel_pool, k),
        f"nDCG@{k}": ndcg_at_k(y_topk, k),
        "PoolSize": int(len(pool_df)),
        "PoolRelevant": total_rel_pool,
        "TopK": int(min(len(g), k)),
        "TopKRelevant": int(y_topk.sum()),
    }


In [102]:
rerank_eval = []

In [103]:
for kind in retriver_target:
    total_df_target = total_df[total_df.ticker==kind]
    rerank_df_target = rerank_df[rerank_df.ticker==kind]
    rerank_eval.append(eval_reranker_chunk(total_df_target,rerank_df_target))

total_rel_pool : 191
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 94
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 0 0 1 1 1 0 1 0 1 1 1 1 1 0 1 1
 1 1 1 1 1 1 1 1 1 0 1 0 1]
total_rel_pool : 119
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 110
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 0 1 1 1 1 1 0 0 1 1]
total_rel_pool : 49
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 0 0 1 1
 0 0 0 0 0 0 1 0 0 0 0 1 0]
total_rel_pool : 122
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 0 1 1]
total_rel_pool : 60
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 0 1
 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 949
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1

In [104]:
rerank_eval_con = dict(zip(Custer_Having_ticker_lst,rerank_eval))

In [105]:
pd.DataFrame(rerank_eval_con)

,우리금융지주,삼성화재,삼성생명,카카오뱅크,메리츠금융지주,현대모비스,현대오토에버,SK하이닉스,현대글로비스
Precision@50,1.00000,0.840000,0.980000,0.920000,0.720000,0.980000,0.960000,1.000000,0.909091
Recall@50(pool),0.26178,0.446809,0.411765,0.418182,0.734694,0.401639,0.800000,0.052687,1.000000
MRR@50,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
MAP@50,1.00000,0.767898,0.966335,0.895437,0.714515,0.979192,0.938764,1.000000,0.966561
nDCG@50,1.00000,0.980836,0.997134,0.994110,0.994518,0.999858,0.995489,1.000000,0.991779
PoolSize,219.00000,123.000000,122.000000,193.000000,66.000000,128.000000,63.000000,968.000000,22.000000
PoolRelevant,191.00000,94.000000,119.000000,110.000000,49.000000,122.000000,60.000000,949.000000,20.000000
TopK,50.00000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,22.000000
TopKRelevant,50.00000,42.000000,49.000000,46.000000,36.000000,49.000000,48.000000,50.000000,20.000000


# 요약문 실험 테스트 수행

## 1. original 본문 가져오기

In [297]:
test_ticker = '우리금융지주'

In [298]:
CrossEncoder_prompt_test = f'''
이 뉴스들 중에서 "{test_ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
다른 회사 언급이 많거나, {test_ticker}가 단순히 함께 언급된 기사라면 제외해줘.
'''

In [91]:
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
retriever_test = vectordb.as_retriever(search_kwargs={"k": 50,'filter' : {'ticker' : test_ticker}})
compressor = CrossEncoderReranker(model=model, top_n=20)

In [313]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever_test
)

In [315]:
compressed_docs = compression_retriever.invoke(CrossEncoder_prompt_test)

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [24]:
def get_full_article_from_chroma(original_idx: int, kind : str, vectordb) -> dict:
    """original_idx 기준으로 chunk들을 모아 원문 복원"""
    # 1. 해당 article의 모든 chunk 가져오기
    VecDB = vectordb._collection
    results = VecDB.get(
      where = { "$and" : [ # 빈 쿼리로 전체 탐색
            {"original_idx": original_idx}, 
            {"ticker": kind}
            ]
         },
         include = ['documents','metadatas']   
        )
        
    if not results:
        return {"error": f"No chunks found for original_idx {original_idx}"}

    #print(results)
    #print(results['metadatas'][0]['chunk_idx'])

    # 4. 대표 metadata 하나 뽑아 저장
    return {
        "title": results['metadatas'][0]["title"],
        "url": results['metadatas'][0]["url"],
        "Date": results['metadatas'][0]["Date"],
        "ticker": results['metadatas'][0]["ticker"],
        "content": results['documents'][0]
    }

In [317]:
original_idxs = list(map(lambda x: x.metadata['original_idx'], compressed_docs))

In [318]:
top_n_original = [get_full_article_from_chroma(idx,kind=test_ticker,vectordb=vectordb) for idx in original_idxs]

In [320]:
top_n_original[:5]

[{'title': "'우리금융지주' 52주 신고가 경신, 전일 외국인 대량 순매수",
  'url': 'https://www.hankyung.com/article/202507145874L',
  'Date': '2025-07-14',
  'ticker': '우리금융지주',
  'content': '◆ 주체별 매매동향- 전일 외국인 대량 순매수지난 한달을 기준으로 보면 외국인이 296.2만주를 순매수했고, 개인들도 41.8만주를 순매수했다. 하지만 기관은 294.9만주를 순매도했다. 같은 기간 이 종목의 거래 비중은 외국인과 개인이 각각 51.3%, 26.2%로 비중이 높다.더욱이 전일 외국인이 대량 순매수를 하고 있어 투자자들의 관심이 집중되고 있다.[그래프]우리금융지주 외국인/기관 매매동향\n\n\n\n\n\n\n\n\n\n\n\n◆ 최근 애널리스트 분석의견- 갈수록 돋보일 고배당 매력 - NH투자증권, BUY07월 09일 NH투자증권의 정준섭 애널리스트는 우리금융지주에 대해 "\'25년 2분기 지배순이익은 다소 부진할 예정. 경상 실적(이자이 익, 비이자이익)은 견조하나, 책준형 신탁 충당금 등으로 대손비용률 상승 (65bp, +20bp q-q, +21bp y-y)이 예상되기 때문. 우리금융지주는 하반기 자사주 매입 기대감은 낮지만, 배당 매력은 갈수록 돋보일 전망"이라고 분석하며, 투자의견 \'BUY\', 목표주가 \'29,000원\'을 제시했다.한경로보뉴스이 기사는 한국경제신문과 금융 AI 전문기업 씽크풀이 공동 개발한 기사 자동생성 알고리즘에 의해 실시간으로 작성된 것입니다.'},
 {'title': "'우리금융지주' 52주 신고가 경신, 전일 외국인 대량 순매수",
  'url': 'https://www.hankyung.com/article/202507084010L',
  'Date': '2025-07-08',
  'ticker': '우리금융지주',
  'content': '◆ 주체별 매매동향- 전일 외국인 대량 순매수지난 한달을 기준으로 보면 외

In [321]:
original_contents = list(map(lambda x: x['content'],top_n_original))

In [322]:
total_contents = ''.join(original_contents)

In [323]:
total_contents

'◆ 주체별 매매동향- 전일 외국인 대량 순매수지난 한달을 기준으로 보면 외국인이 296.2만주를 순매수했고, 개인들도 41.8만주를 순매수했다. 하지만 기관은 294.9만주를 순매도했다. 같은 기간 이 종목의 거래 비중은 외국인과 개인이 각각 51.3%, 26.2%로 비중이 높다.더욱이 전일 외국인이 대량 순매수를 하고 있어 투자자들의 관심이 집중되고 있다.[그래프]우리금융지주 외국인/기관 매매동향\n\n\n\n\n\n\n\n\n\n\n\n◆ 최근 애널리스트 분석의견- 갈수록 돋보일 고배당 매력 - NH투자증권, BUY07월 09일 NH투자증권의 정준섭 애널리스트는 우리금융지주에 대해 "\'25년 2분기 지배순이익은 다소 부진할 예정. 경상 실적(이자이 익, 비이자이익)은 견조하나, 책준형 신탁 충당금 등으로 대손비용률 상승 (65bp, +20bp q-q, +21bp y-y)이 예상되기 때문. 우리금융지주는 하반기 자사주 매입 기대감은 낮지만, 배당 매력은 갈수록 돋보일 전망"이라고 분석하며, 투자의견 \'BUY\', 목표주가 \'29,000원\'을 제시했다.한경로보뉴스이 기사는 한국경제신문과 금융 AI 전문기업 씽크풀이 공동 개발한 기사 자동생성 알고리즘에 의해 실시간으로 작성된 것입니다.◆ 주체별 매매동향- 전일 외국인 대량 순매수지난 한달을 기준으로 보면 외국인이 490.4만주를 순매수했고, 개인들도 15.8만주를 순매수했다. 하지만 기관은 470.8만주를 순매도했다. 같은 기간 이 종목의 거래 비중은 외국인과 개인이 각각 52.6%, 24.3%로 비중이 높다.한편 외국인은 이 종목에 대해서 최근 3일 연속 29.4만주 순매수를 하고 있다. 더욱이 전일 외국인이 대량 순매수를 하고 있어 투자자들의 관심이 집중되고 있다.[그래프]우리금융지주 외국인/기관 매매동향\n\n\n\n\n\n\n\n\n\n\n\n◆ 최근 애널리스트 분석의견- 2Q25 Preview: 시간이 더 필요하다 - 상상인증권, BUY(신규)07월 03일 상상인증권의 김현수 애널리스트는 우리

In [324]:
len(total_contents)

22496

수기 입력으로 확인
> 확인 사유 : 0번째 문서가 이상하..?

In [310]:
retriever = vectordb.as_retriever(search_kwargs={"k": 50,'filter' : {'ticker' : test_ticker}})
CrossEncoder_prompt = f'''
이 뉴스들 중에서 "{test_ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
다른 회사 언급이 많거나, {test_ticker}가 단순히 함께 언급된 기사라면 제외해줘.
'''
print(CrossEncoder_prompt.strip())
raw_docs = retriever.get_relevant_documents(CrossEncoder_prompt)
pairs = [(CrossEncoder_prompt, d.page_content) for d in raw_docs]

scores = model.score(pairs)
scored = sorted(zip(raw_docs, scores), key=lambda x: float(x[1]), reverse=True)[:50]
scored_docs=[]
# 3) 점수 붙이고 재정렬
for d, s in zip(raw_docs, scores):
    dd = deepcopy(d)
    dd.metadata["relevance_score"] = float(s)
    scored_docs.append(dd)

이 뉴스들 중에서 "우리금융지주"가 핵심 주제로 다뤄진 기사만 알려줘.
다른 회사 언급이 많거나, 우리금융지주가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [311]:
pd.DataFrame(list(map(lambda x: x.metadata,scored_docs))).sort_values(by=['ticker','relevance_score'],ascending=False).head(5)

,chunk_idx,title,url,label,original_idx,ticker,Date,relevance_score
3,0,"'우리금융지주' 52주 신고가 경신, 전일 외국인 대량 순매수",https://www.hankyung.com/article/202507145874L,1,237,우리금융지주,2025-07-14,0.602175
14,0,"'우리금융지주' 52주 신고가 경신, 전일 외국인 대량 순매수",https://www.hankyung.com/article/202507084010L,1,374,우리금융지주,2025-07-08,0.467356
9,0,"""우리금융지주, 연말로 갈수록 고배당 부각…목표가↑""-NH",https://www.hankyung.com/article/2025070836976,1,476,우리금융지주,2025-07-08,0.397002
6,1,대출 수익성 악화에…4대 금융 실적 꺾였다,https://www.hankyung.com/article/2025071501731,1,158,우리금융지주,2025-07-15,0.358331
10,0,"“금리 하락에도 이익 증가” 금융지주, 실적 온도차",https://magazine.hankyung.com/business/article...,1,446,우리금융지주,2025-07-21,0.265313


## 요약하기

### 요약함수 호출
- 가져온 원 본문을 전부 적용하기

In [27]:
from summart_function_openai_2 import NewsSummaryAgent, build_summary_graph  # 너가 만든 것

In [28]:
def summarize_top_articles_2(total_contents: str,ticker:str,max_iters:int) -> pd.DataFrame:
    agent = NewsSummaryAgent(max_iters=max_iters)
    runnable = build_summary_graph(agent)

    rows = []
    doc = total_contents

    state = {"article": doc, "summary": "", "feedback": "", "iteration": 0}
    result = runnable.invoke(state)

    print("✅ 실행 결과 키:", result.keys())
    # 여기서 final_summary 반드시 존재해야 함(위 패치 기준)
    final_summary = result.get("final_summary")
    if not final_summary:
        print("❌ final_summary 없음. 디버그용 전체 상태:", result)
        # 계속 진행할지, 실패로 표기할지 선택

    rows.append({
        "ticker": ticker,
        "date": '2025-07-30', # 위 조회 기준일자로 연동시켜서 바꿀 예정
        "summary": final_summary,
        "feedback": result.get("last_feedback", "피드백 없음"),
    })

    return pd.DataFrame(rows)


In [325]:
result_df = summarize_top_articles_2(total_contents,ticker=test_ticker,max_iters=5)

summart : ✅ 주요 요약
- 외국인, 우리금융지주 대량 순매수
  외국인은 최근 3일 연속으로 우리금융지주 주식을 대량 순매수하며 투자자들의 관심이 집중되고 있다.

- 우리금융지주, 2분기 순이익 감소 전망
  우리금융지주의 2분기 순이익은 전년 대비 감소할 것으로 예상되며, 이는 모바일 트레이딩 시스템 개발과 신규 인력 채용 등으로 인한 판매관리비 증가가 원인으로 분석된다.

- 4대 금융지주, 비이자이익 증가로 실적 선방
  KB, 신한, 하나, 우리 등 4대 금융지주는 비이자이익 증가 덕분에 2분기 실적이 개선되었으나, 하반기에는 경기 침체와 대출 자산 확대 어려움으로 실적이 나빠질 우려가 있다.

🔑 키워드: 외국인 순매수, 우리금융지주, 2분기 순이익, 비이자이익, 4대 금융지주, 경기 침체, 대출 자산, 실적 전망.
[should_stop] Iteration: 0
[should_stop] Feedback:
 - 정확성: 부족함 <reason> [요약에서 언급된 외국인 순매수와 관련된 정보는 기사에 있지만, 요약에서 언급된 '우리금융지주 2분기 순이익 감소 전망'의 구체적인 원인인 '모바일 트레이딩 시스템 개발과 신규 인력 채용'은 기사에 명시되어 있지 않습니다.] </reason>
- 포괄성: 부족함 <reason> [요약은 외국인 순매수와 2분기 실적 전망에 대한 주요 내용을 다루고 있지만, 기사에 포함된 다양한 애널리스트 의견과 금융지주들의 배당 및 자사주 매입 관련 정보가 누락되었습니다.] </reason>
- 간결성: 좋음 <reason> [요약은 불필요한 표현 없이 핵심 정보를 간결하게 전달하고 있습니다.] </reason>
- 문장구성: 좋음 <reason> [문장이 자연스럽고 명확하게 구성되어 있어 이해하기 쉽습니다.] </reason>
- 일관성: 부족함 <reason> [요약은 특정 종목인 '우리금융지주'에 대한 내용에 집중하고 있지만, 기사에는 다른 금융지주들에 대한 정보도 포함되어 있어 일관성이 부족합니다.] </reas

In [326]:
result_df

,ticker,date,summary,feedback
0,우리금융지주,2025-07-30,"✅ 주요 요약\n\n- **외국인, 우리금융지주 대량 순매수**\n - 외국인은 ...",- 정확성: 부족함 <reason> [요약에서 언급된 일부 정보가 원문 기사와 일치...


In [327]:
print(result_df.summary.values[0])

✅ 주요 요약

- **외국인, 우리금융지주 대량 순매수**
  - 외국인은 최근 3일 동안 우리금융지주 주식을 29.4만주 순매수하며 투자자들의 주목을 받고 있습니다. 지난 한 달 동안 외국인은 490.4만주를 순매수한 반면, 기관은 470.8만주를 순매도했습니다.

- **우리금융지주, 2분기 순이익 감소 전망**
  - 우리금융지주의 2분기 순이익은 전년 대비 8.6% 감소한 8784억원으로 예상됩니다. 이는 모바일 트레이딩 시스템 개발과 신규 인력 채용에 따른 판매관리비 증가가 주요 원인으로 분석됩니다.

- **4대 금융지주, 비이자이익 증가로 실적 선방**
  - KB, 신한, 하나, 우리 등 4대 금융지주는 비이자이익 증가 덕분에 2분기 실적이 개선되었습니다. 그러나 하반기에는 경기 침체와 대출 자산 확대 어려움으로 실적이 나빠질 우려가 있습니다.

- **애널리스트 의견 및 배당 매력**
  - NH투자증권과 상상인증권은 우리금융지주의 배당 매력을 강조하며 투자의견을 '매수'로 유지했습니다. 하반기 자사주 매입 가능성은 낮지만, 배당 매력은 더욱 부각될 전망입니다.

- **금융지주들의 배당 및 자사주 매입**
  - 4대 금융지주는 하반기 자사주 매입 및 소각 규모가 최소 1조 6천억 원에 이를 것으로 예상되며, 주주친화 정책 강화가 주가 상승의 모멘텀으로 작용하고 있습니다.

- **다른 금융지주사들의 실적 및 전망**
  - 신한금융지주는 2분기 순이익이 1.3% 증가할 것으로 예상되며, 하나금융지주는 7% 이상 증가할 것으로 보입니다. 이는 신용카드, 증권 중개, 운용리스 등 수수료 수익이 양호한 흐름을 이어가고 있기 때문입니다.

- **금융지주사들의 장기 전망**
  - 4대 금융지주의 연간 순이익은 총 18조원에 육박할 것으로 예상되며, 이는 비이자이익 증가와 주주환원 정책 강화에 기인합니다. 그러나 경기 침체와 대출 규제 강화로 인해 하반기 실적에는 불확실성이 존재합니다.

🔑 **키워드**: 외국인 순매수, 우리금융지주, 2분기 

## 피드백 정리
- 삼성전자  : 삼성전자에 대한 내용 잘 요약
- 우리금융지주 : 우리금융지주에 대한 내용 잘 요약
- SK하이닉스 : 요약은 되었으나,, 시장 내/외부 사람들의 구매 패턴, 동향에 대한 정보가 주로 요약이 되고 있음<br>
            > recall@50(pool) 이 상당히 낮게 나옴!
  > SK하이닉스에 대한 케이스를 봤을 때, chunk에 대해서만 요약을 해야하나 싶음<br>
  > 2025.08.11) 현재 작업은 전체 원문에 대한 요약이었음. 그래서.. SK 하이닉스 회사 본질에 대한 정보를 제대로 요약해주지 못하나 싶음<br>
  > 프롬프트 내부 요약 평가 기준 의 "일관성" 영역에 명확히 종목명을 말해야겠다..

## 시뮬레이션 한 애들 전체 돌리기

In [52]:
total_con_lst = []

In [6]:
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [54]:
for ticker in Custer_Having_ticker_lst:
    print(f'{ticker} =======')
    CrossEncoder_prompt_test = f'''
    이 뉴스들 중에서 "{ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, {ticker}가 단순히 함께 언급된 기사라면 제외해줘.
    '''
    retriever_test = vectordb.as_retriever(search_kwargs={"k": 50,'filter' : {'ticker' : ticker}})
    compressor = CrossEncoderReranker(model=model, top_n=20)
    compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever_test
    )
    compressed_docs = compression_retriever.invoke(CrossEncoder_prompt_test)
    original_idxs = list(map(lambda x: x.metadata['original_idx'], compressed_docs))
    top_n_original = [get_full_article_from_chroma(idx,kind=ticker,vectordb=vectordb) for idx in original_idxs]
    print(f'원문 개수 : {len(top_n_original)}')
    original_contents = list(map(lambda x: x['content'],top_n_original))
    total_contents = ''.join(original_contents)
    print(f'원문 총 length : {len(total_contents)}')
    total_con_lst.append(summarize_top_articles_2(total_contents,ticker=ticker,max_iters=5))

우리금융지주 =======


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


원문 개수 : 20
원문 총 length : 17384
summart : ✅ 주요 요약
- 글로벌 투자자, 한국 금융주에 대한 관치금융 리스크 우려
  글로벌 투자자들이 한국 금융주에 대한 정부의 관치금융 리스크 확산을 우려하며, 주가 하락에 대한 문의가 증가하고 있음.

- 정부의 금융권 압박, 은행주 주가 하락의 주요 원인
  정부의 상생 압박과 세금, 과징금 증가가 은행주 주가에 부정적 영향을 미치며, 주주환원 정책의 동력을 상실할 수 있다는 우려가 제기됨.

- 금융사 평균 급여 증가, 고액 연봉 논란
  금융사 직원들의 평균 급여가 증가하면서 고액 연봉 논란이 일고 있으며, 이는 정부의 금융권 압박과 맞물려 상생 압박이 더욱 거세질 가능성이 있음.

📌 키워드: 관치금융 리스크, 글로벌 투자자, 은행주 주가 하락, 정부 압박, 고액 연봉 논란, 금융권 상생 압박, 주주환원 정책.
[should_stop] Iteration: 0
[should_stop] Feedback:
 - 정확성: 좋음
- 포괄성: 부족함 <reason> [요약에서 언급된 '금융사 평균 급여 증가'와 관련된 부분은 원문에서 다루고 있지 않음. 또한, 원문에서 언급된 외국인 투자자들의 방한 및 금융당국과의 회동 계획 등 구체적인 내용이 요약에 포함되지 않음.] </reason>
- 간결성: 좋음
- 문장구성: 좋음
- 일관성: 부족함 <reason> [요약에서 언급된 '금융사 평균 급여 증가' 부분이 원문 기사와 관련이 없으며, 특정 종목에 대한 내용이 아닌 일반적인 금융권 상황을 다루고 있음.] </reason>

피드백:
1. 포괄성: 요약에 포함된 '금융사 평균 급여 증가' 부분은 원문 기사에서 다루고 있지 않으므로, 이 부분을 삭제하고 원문에서 다룬 외국인 투자자들의 방한 및 금융당국과의 회동 계획 등 구체적인 내용을 추가해야 합니다.
2. 일관성: 요약은 특정 종목에 대한 내용이 아니라 금융권 전반에 대한 내용을 다루고 있으므로, 요약의 일관성을 높이기 위해 원문 기사에 충실

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


원문 개수 : 20
원문 총 length : 35595
summart : ✅ 주요 요약
- 삼성화재, AI 의료심사 도입으로 암 진단 및 수술급여 심사 효율성 향상
  삼성화재는 AI 기반의 의료심사 시스템을 도입하여 암 진단 및 수술급여 심사의 효율성과 정확성을 높였으며, 인력 검토 비중이 약 55% 감소했다고 발표했습니다.

- '해외 2시간 항공지연 특약' 출시로 항공기 지연 및 결항 손해 보장
  삼성화재는 해외 출발 항공편의 2시간 이상 지연 및 결항 시 발생하는 손해를 최대 50만원까지 보장하는 '해외 2시간 항공지연 특약'을 출시하여 고객의 여행 불편을 최소화하고자 했습니다.

- 외국인 투자자, SK하이닉스 등 주요 종목 순매수
  외국인 투자자들은 SK하이닉스, 현대차, HD한국조선해양 등 주요 종목을 집중적으로 매수하였으며, 운수장비 업종에 속한 종목들이 다수 포함되었습니다.

키워드: 삼성화재, AI 의료심사, 항공지연 특약, 외국인 투자자, SK하이닉스, 현대차, HD한국조선해양
[should_stop] Iteration: 0
[should_stop] Feedback:
 - 정확성: 좋음
  <reason> 요약 내용이 원문 기사와 일치하며, 주요 정보가 정확하게 전달되었습니다. </reason>

- 포괄성: 좋음
  <reason> 기사에서 다루는 주요 주제인 삼성화재의 AI 의료심사 도입, 항공지연 특약 출시, 외국인 투자자의 주요 종목 매수에 대한 정보가 모두 포함되어 있습니다. </reason>

- 간결성: 좋음
  <reason> 요약은 불필요한 표현 없이 간결하게 작성되었습니다. </reason>

- 문장구성: 좋음
  <reason> 문장이 자연스럽고 명확하게 구성되어 있습니다. </reason>

- 일관성: 좋음
  <reason> 특정 종목에 대한 내용만 존재하며, 다른 불필요한 정보가 포함되지 않았습니다. </reason>

피드백:
현재 요약은 모든 평가 기준에서 좋은 평가를 받았습니다. 각 항목에 대한 정보가 

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


KeyboardInterrupt: 

In [31]:
total_df = pd.concat(total_con_lst)

In [32]:
total_df.to_csv('total_df.csv',encoding='utf-8-sig')

In [33]:
for _,data in total_df.iterrows():
    print(data.ticker)
    print(data.summary)
    print('-'*100)

우리금융지주
✅ 주요 요약

- **외국인 투자자, 한국 금융시장에 대한 우려 증가**
  글로벌 투자은행인 JP모간과 모건스탠리가 한국을 방문하여 금융당국 및 주요 금융지주 회장들과 회동할 예정입니다. 이는 정부의 금융권 압박과 관련된 리스크를 직접 평가하기 위한 것으로, 국민성장펀드와 교육세 인상 등으로 인한 금융사의 건전성 악화에 대한 우려가 커지고 있습니다.

- **정부의 금융권 압박, 은행주 하락 원인**
  정부의 정책 변화, 특히 교육세 인상과 금융권에 대한 비판이 은행주의 투자심리를 위축시키고 있습니다. 이는 외국인 투자자들이 한국 금융시장에서의 투자 매력을 재평가하게 만들고 있으며, 금융지주 주가에 부정적인 영향을 미치고 있습니다.

- **금융권의 높은 연봉, 정부의 상생 압박 강화**
  금융권의 높은 연봉과 사상 최대 실적이 정부의 상생 압박을 강화시키고 있습니다. 이는 금융사의 수익성과 경쟁력을 약화시키는 요인으로 작용하고 있으며, 외국인 지분율이 높은 금융지주사들은 비상 상황에 직면하고 있습니다.

🔑 **키워드**: 글로벌 투자자, 금융시장, 정부 압박, 은행주 하락, 금융권 연봉, 상생 압박, 교육세 인상, 투자심리, 금융사 수익성.

**개선 포인트**:
1. **정확성**: 기사에 명시된 내용을 중심으로 요약을 구성하여 정확성을 높였습니다.
2. **간결성**: 중복된 표현을 제거하고, 핵심 내용을 중심으로 간결하게 작성했습니다.
3. **일관성**: 금융권의 연봉 문제와 정부의 정책 변화에 대한 영향을 별도로 다루어 일관성을 유지했습니다.
----------------------------------------------------------------------------------------------------
삼성화재
✅ 주요 요약
- 삼성생명, 상반기 사상 최대 실적 기록
  삼성생명은 올해 상반기 순이익이 1조3941억원으로 전년 동기 대비 1.9% 증가하며, 2년 연속 최대 실적을 기록했습니다. 건강보험 판매 호

## 2. chunk만 수행하기

In [ ]:
test_ticker = '우리금융지주'
CrossEncoder_prompt_test = f'''
이 뉴스들 중에서 "{test_ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
다른 회사 언급이 많거나, {test_ticker}가 단순히 함께 언급된 기사라면 제외해줘.
'''

In [254]:
#model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
compressor = CrossEncoderReranker(model=model, top_n=50)

In [255]:
retriever_test = vectordb.as_retriever(search_kwargs={"k": 50,'filter' : {'ticker' : test_ticker}})

In [256]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever_test
)

In [257]:
compressed_docs = compression_retriever.invoke(CrossEncoder_prompt)

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [259]:
list(map(lambda x: x.metadata['title'],compressed_docs))[:5]

["SK하이닉스 또 '사상 최고' 실적…분기 영업익 9조 넘었다",
 "SK하이닉스 사상 최대 실적에도…'신중론' 여전한 이유 [종목+]",
 "'역대급 실적' SK하이닉스, 2분기 영업이익 9조원 넘었다",
 'AI 수요 폭증과 HBM 기술 주도에 힘입은 SK하이닉스, 주가 반등 흐름 가속화',
 'SK하이닉스, 삼성전자 제치고 ‘대학생이 일하고 싶은 기업’ 첫 1위']

compressed_docs와 수기로 계산한 것이 같은 것 증명

In [246]:
retriever = vectordb.as_retriever(search_kwargs={"k": 50,'filter' : {'ticker' : test_ticker}})
CrossEncoder_prompt = f'''
이 뉴스들 중에서 "{test_ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
다른 회사 언급이 많거나, {test_ticker}가 단순히 함께 언급된 기사라면 제외해줘.
'''
print(CrossEncoder_prompt.strip())
raw_docs = retriever.get_relevant_documents(CrossEncoder_prompt)
pairs = [(CrossEncoder_prompt, d.page_content) for d in raw_docs]

scores = model.score(pairs)
scored = sorted(zip(raw_docs, scores), key=lambda x: float(x[1]), reverse=True)[:50]
scored_docs=[]
# 3) 점수 붙이고 재정렬
for d, s in zip(raw_docs, scores):
    dd = deepcopy(d)
    dd.metadata["relevance_score"] = float(s)
    scored_docs.append(dd)

이 뉴스들 중에서 "SK하이닉스"가 핵심 주제로 다뤄진 기사만 알려줘.
다른 회사 언급이 많거나, SK하이닉스가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [260]:
pd.DataFrame(list(map(lambda x: x.metadata,scored_docs))).sort_values(by=['ticker','relevance_score'],ascending=False).head(5)

,original_idx,title,url,label,chunk_idx,ticker,Date,relevance_score
47,3560,SK하이닉스 또 '사상 최고' 실적…분기 영업익 9조 넘었다,https://www.hankyung.com/article/202507248226g,1,1,SK하이닉스,2025-07-24,0.684618
25,3927,SK하이닉스 사상 최대 실적에도…'신중론' 여전한 이유 [종목+],https://www.hankyung.com/article/2025072510306,1,1,SK하이닉스,2025-07-25,0.672207
37,3724,"'역대급 실적' SK하이닉스, 2분기 영업이익 9조원 넘었다",https://magazine.hankyung.com/business/article...,1,0,SK하이닉스,2025-07-24,0.611750
15,3744,"AI 수요 폭증과 HBM 기술 주도에 힘입은 SK하이닉스, 주가 반등 흐름 가속화",https://www.hankyung.com/article/202507313103a,1,0,SK하이닉스,2025-07-31,0.601519
14,3867,"SK하이닉스, 삼성전자 제치고 ‘대학생이 일하고 싶은 기업’ 첫 1위",https://magazine.hankyung.com/job-joy/article/...,1,2,SK하이닉스,2025-07-28,0.455363


이어서 시작

In [336]:
unique_lst = list(set(list(map(lambda x: x.page_content,compressed_docs))))

In [337]:
chunk_total = ''.join(unique_lst)

In [338]:
chunk_total

'KB 신한 하나 우리 등 4대 금융지주의 올해 2분기 합산 순이익이 전년 동기 대비 감소한 것으로 파악됐다. 금융지주의 실적이 전년 동기 대비 감소한 것은 홍콩 H지수 주가연계증권(ELS) 배상으로 1조원 넘는 일회성 비용이 발생한 2024년 1분기를 제외하면 1년 반 만에 처음이다. 금융회사의 핵심 수익원인 이자수익이 줄줄이 감소한 결과다. 금융지주의 핵심 자회사인 은행들이 가계대출 억제 정책과 경기 침체로 대출 자산을 확대하기 어려운 만큼 향후 금융지주의 실적 감소세가 본격화할 것이란 관측이 제기된다.\n                    \n\n\n\n\n\n\n이미지 크게보기증가로 순이익이 지난해 3조 1715억원에서 올해 3조 1095억원으로 소폭 줄어들 것으로 예상된다.4대 금융지주의 순이익 합계는 지난해 16조 5268억원에서 올해 17조 8250억원으로 8% 가까이 증가할 전망이다.정부의 고강도 부동산 대출 규제에도 불구하고 주요 금융지주들의 연간 실적 전망은 오히려 상향 조정되는 추세다.이는 각 금융지주가 이자이익 축소에 대비해 새로운 수익원 확보에 주력했기 때문으로 분석된다.KB금융은 오는 24일, 신한·하나·우리금융은 25일 차례로 2분기 실적을 발표할 예정이다.정유진 기자 jinjin@hankyung.com강세장을 이끈 것은 5조원 순매수한 연기금 외에 자사주와 외국인이다. 현재까지 자사주 매입은 약 10조원에 달할 것으로 추정되는데 삼성전자와 은행이 주요 매수 주체였다. 이 추세는 정부 정책과 맞물려서 하반기에도 이어진다. 하반기 6조원이 더해지면 올해 약16조원이 자사주 매입의 형태로 증시에 유입되는데 이는 작년의 2조원 자사주 순매수와 크게 대비된다. 외인은 연초부터 4월 말까지 트럼프의 관세 악재로 18조 매도했지만 이후 매수로 전환해서 12조원의 순매수를 누적했다.2000년 이후 25년 동안 외인의 누적 순매수 추세는 2번의 80조원 싸이클과 1번의 40조원 사이클을 보인 후 4번째 순환사이클을 막 시작했다. 직전 순매수 저점에

In [339]:
len(chunk_total)

37824

In [340]:
result_df_chunk = summarize_top_articles_2(chunk_total,ticker=test_ticker,max_iters=5)

summart : ✅ 주요 요약
- 4대 금융지주, 2분기 순이익 전년 대비 감소
  KB, 신한, 하나, 우리 등 4대 금융지주의 2분기 합산 순이익이 전년 동기 대비 감소했으며, 이는 이자수익 감소와 가계대출 억제 정책으로 인한 결과로 분석됨.

- 금융지주, 주주환원 정책 강화
  4대 금융지주는 자사주 매입 및 소각을 포함한 주주환원 정책을 강화하고 있으며, 이는 주가 상승과 주주가치 제고에 긍정적인 영향을 미칠 것으로 예상됨.

- 정부의 부동산 대출 규제와 증시 친화 정책
  정부의 부동산 대출 규제에도 불구하고 금융지주들의 실적 전망은 상향 조정되고 있으며, 이는 새로운 수익원 확보와 증시 친화적인 정책 덕분으로 분석됨.

🔑 키워드
- 4대 금융지주
- 순이익 감소
- 이자수익
- 가계대출 억제
- 주주환원 정책
- 자사주 매입
- 부동산 대출 규제
- 증시 친화 정책
[should_stop] Iteration: 0
[should_stop] Feedback:
 - 정확성: 부족함 <reason> [요약 내용이 원문 기사와 일부 일치하지 않으며, 특히 금융지주의 순이익 증가에 대한 부분이 잘못 전달됨] </reason>
- 포괄성: 부족함 <reason> [기사의 중요한 정보 중 일부가 누락되었으며, 특히 정부의 정책 변화와 이에 따른 금융지주의 대응 전략이 충분히 다루어지지 않음] </reason>
- 간결성: 좋음 <reason> [요약은 간결하게 작성되어 있으며, 불필요한 정보가 포함되지 않음] </reason>
- 문장구성: 좋음 <reason> [문장이 자연스럽고 명확하게 구성되어 있음] </reason>
- 일관성: 부족함 <reason> [특정 종목에 대한 내용이 아닌, 전체 금융지주에 대한 내용임에도 불구하고 일관성이 부족함] </reason>

피드백:
1. 정확성을 높이기 위해 원문 기사에서 언급된 순이익 증가와 감소에 대한 정보를 정확히 반영해야 합니다. 특히, 금융지주의 순이익이 전년 대비 증가했다는 부분을 명확히 해야 합니다

KeyboardInterrupt: 

In [270]:
print(result_df_chunk.summary.values[0])

✅ 주요 요약

- **SK하이닉스, 2분기 사상 최대 실적 기록**
  SK하이닉스는 2분기 매출 22조2320억원, 영업이익 9조2129억원을 기록하며 사상 최대 실적을 달성했다. 이는 AI 메모리 수요 증가와 D램 및 낸드플래시 출하량 증가에 기인한다.

- **HBM 시장에서의 경쟁 심화 우려**
  SK하이닉스는 HBM 시장에서의 독점적 지위가 경쟁 심화로 인해 위협받을 수 있다는 우려가 제기되고 있다. 삼성전자와 마이크론의 시장 진입이 예상되며, 가격 하락 가능성이 논의되고 있다.

- **SK하이닉스의 HBM 수요 성장 확신**
  SK하이닉스는 HBM 수요가 지속적으로 증가할 것이라고 확신하며, AI 시장에서의 핵심 제품으로서의 중요성을 강조하고 있다. 이에 따라 HBM 생산을 위한 투자를 확대할 계획이다.

- **주가 변동 및 외부 기관의 평가**
  SK하이닉스의 실적 발표 이후 주가는 일시적으로 상승했으나, 외부 기관의 평가에 따라 변동성을 보였다. 골드만삭스는 HBM 시장의 경쟁 심화로 인한 가격 하락 가능성을 지적하며 투자의견을 하향 조정했다. 반면, 다른 기관들은 SK하이닉스의 기술력과 시장 지배력을 긍정적으로 평가하며, 장기적인 성장 가능성을 강조했다.

🔑 키워드
- SK하이닉스
- 2분기 실적
- HBM 시장
- AI 메모리
- 경쟁 심화
- 가격 하락
- 투자 확대
- 주가 변동
- 외부 기관 평가


## 청크 토탈 시뮬레이션

In [34]:
chunk_array_lst = []

In [51]:
for ticker in Custer_Having_ticker_lst:
    print(f'{ticker} 진행=======================')
    CrossEncoder_prompt_test = f'''
    이 뉴스들 중에서 "{ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, {ticker}가 단순히 함께 언급된 기사라면 제외해줘.
    '''
    retriever_test = vectordb.as_retriever(search_kwargs={"k": 50,'filter' : {'ticker' : ticker}})
    compressor = CrossEncoderReranker(model=model, top_n=20)
    compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever_test
    )
    compressed_docs = compression_retriever.invoke(CrossEncoder_prompt_test)
    unique_lst = list(set(list(map(lambda x: x.page_content,compressed_docs))))
    print(f'chunk 개수 : {len(unique_lst)}')
    chunk_total = ''.join(unique_lst)
    chunk_array_lst.append(summarize_top_articles_2(chunk_total,ticker=ticker,max_iters=5))

우리금융지주 진행=======================


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


chunk 개수 : 7
summart : ✅ 주요 요약
- 국내 4대 금융지주, 상반기 평균 급여 1억원 돌파
  KB금융, 우리금융 등 주요 금융지주들이 사상 최대 실적을 기록하며 평균 급여가 1억원을 넘었고, 이는 제조업 대기업보다 높은 수준이다.

- 금융주 주가 하락, 정부의 압박과 세금 인상 영향
  정부의 교육세 인상 및 과징금 부과 등으로 인해 금융주 주가가 하락하고 있으며, 외국인 주주 비율이 높은 금융지주사들이 특히 영향을 받고 있다.

- 정부의 금융권 압박, 외국인 투자자 우려 증가
  이재명 정부의 금융권에 대한 압박이 강화되면서 외국인 투자자들 사이에서 우려가 커지고 있으며, 이는 금융지주사의 주가에 부정적인 영향을 미치고 있다.

키워드: 금융지주, 평균 급여, 주가 하락, 정부 압박, 교육세 인상, 외국인 투자자, 이재명 정부, 금융권 압박, 주주환원.
[should_stop] Iteration: 0
[should_stop] Feedback:
 - 정확성: 좋음
- 포괄성: 부족함 <reason> [요약이 주요 내용을 잘 담고 있지만, 기사에서 언급된 비금융권의 급여 정보나 코스피 상장사의 실적 개선 등 추가적인 세부 사항이 누락되었습니다.] </reason>
- 간결성: 좋음
- 문장구성: 좋음
- 일관성: 좋음

피드백:
포괄성 측면에서, 요약은 금융지주와 관련된 주요 내용을 잘 전달하고 있지만, 기사에서 언급된 다른 중요한 정보들, 예를 들어 비금융권의 급여 정보나 코스피 상장사의 실적 개선 등의 내용도 포함하면 더 포괄적인 요약이 될 것입니다. 이러한 세부 사항을 추가하여 독자가 전체 기사의 맥락을 더 잘 이해할 수 있도록 하는 것이 좋습니다.
[should_stop] next_step = no
====== result : refined_summary='✅ 주요 요약\n\n- **국내 4대 금융지주, 상반기 평균 급여 1억원 돌파**  \n  KB금융과 우리금융 등 주요 금융지주들이 사상 최대 실적을 기록하며 평균 급여가 1억

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


KeyboardInterrupt: 

In [36]:
chunk_df = pd.concat(chunk_array_lst)

In [38]:
chunk_df.to_csv('chunk_df.csv',encoding='utf-8-sig')

In [39]:
chunk_df

,ticker,date,summary,feedback
0,우리금융지주,2025-07-30,"✅ 주요 요약\n\n- **국내 4대 금융지주, 평균 급여 사상 첫 반기 1억 원 ...","- 정확성: 좋음\n <reason> [요약 내용이 원문 기사와 일치하며, 주요 ..."
0,삼성화재,2025-07-30,"✅ 주요 요약\n\n- **삼성화재, AI 의료심사 도입으로 암 진단 심사 효율성 ...",**평가:**\n\n- 정확성: 좋음\n <reason> 요약 내용이 원문 기사와...
0,삼성생명,2025-07-30,"✅ 주요 요약\n\n- **삼성그룹, 하반기 대규모 공개채용 실시** \n 삼성...","- 정확성: 좋음 <reason> [요약 내용이 원문 기사와 일치하며, 삼성그룹의 ..."
0,카카오뱅크,2025-07-30,"✅ **주요 요약**\n\n- **카카오 창업자 김범수, 법적 문제 직면**\n ...","- 정확성: 좋음 <reason> 요약 내용이 원문 기사와 일치하며, 주요 사건과 ..."
0,메리츠금융지주,2025-07-30,"✅ 주요 요약\n\n- **현대해상, 실적 부진에도 주가 상승**\n - 현대해상...","- 정확성: 좋음\n <reason> [요약 내용이 원문 기사와 일치하며, 현대해..."
0,현대모비스,2025-07-30,"✅ 주요 요약\n\n- **현대모비스, 차량용 반도체 연구개발 프로세스 ISO 26...","- 정확성: 좋음\n <reason> [요약 내용이 원문 기사와 일치하며, 현대모..."
0,현대오토에버,2025-07-30,"✅ 주요 요약\n- 기관 투자자, 다양한 종목 순매수\n 기관 투자자들은 KODE...","- 정확성: 부족함 <reason> 요약 내용이 원문 기사와 일부 일치하지만, 원문..."
0,SK하이닉스,2025-07-30,"✅ 주요 요약\n\n- **삼성전자 주가 상승, SK하이닉스 주가 하락** \n ...","- 정확성: 좋음\n <reason> [요약 내용이 원문 기사와 대체로 일치하며,..."
0,현대글로비스,2025-07-30,"✅ 주요 요약\n\n- **현대차와 기아, 외국인 매수세로 주가 상승** \n ...","- 정확성: 좋음\n <reason> 요약 내용이 원문 기사와 일치하며, 주요 정..."
0,현대차,2025-07-30,"✅ 주요 요약\n\n- **현대차그룹, 미국에 50억 달러 추가 투자 발표**\n ...",- 정확성: 부족함 <reason> [요약에서 언급된 몇 가지 세부 사항이 원문 기...


In [40]:
for _,data in chunk_df.iterrows():
    print(data.ticker)
    print(data.summary)
    print('-'*100)

우리금융지주
✅ 주요 요약

- **국내 4대 금융지주, 평균 급여 사상 첫 반기 1억 원 돌파**
  - KB금융이 1억1200만원으로 가장 높은 평균 급여를 기록했으며, 금융지주들의 상반기 순이익이 10조3254억원을 넘어서며 급여 상승에 기여함.

- **금융지주 주가 하락, 정부의 세금 및 과징금 부담 증가**
  - 교육세 인상과 과징금 부과 등으로 인해 금융지주 주가가 하락하고 있으며, 정부의 압박이 주가에 부정적 영향을 미치고 있음.

- **외국인 투자자, 정부의 금융권 압박에 우려**
  - 외국인 지분율이 높은 금융지주사들이 정부의 압박으로 인해 주가가 하락하고 있으며, 외국인 투자자들의 우려가 증가하고 있음.

🔑 **키워드**: 4대 금융지주, 평균 급여, 주가 하락, 교육세 인상, 외국인 투자자, 정부 압박, 금융권, 과징금, KB금융, 신한금융, 하나금융, 우리금융, 이자놀이, 주주환원.
----------------------------------------------------------------------------------------------------
삼성화재
✅ 주요 요약

- **삼성화재, AI 의료심사 도입으로 암 진단 심사 효율성 향상**  
  삼성화재는 AI 의료심사를 도입하여 암 진단 보험금 지급 심사의 효율성과 정확성을 높였습니다. 이 시스템은 의료 데이터를 자동으로 분석하여 심사자의 검토 시간을 단축하고, 인력 검토 비중을 55% 감소시켰습니다. 삼성화재는 AI의료심사 시스템의 특허출원을 완료했습니다.

- **외국인 투자자, SK하이닉스 등 주요 종목 순매수**  
  외국인 투자자들은 SK하이닉스, 현대차, HD한국조선해양, 삼양식품, 기아, 이수페타시스, 현대로템, 한화오션, 한국항공우주, 삼성화재 등 주요 종목을 집중적으로 매수했습니다. 특히 운수장비 업종의 종목들이 다수 포함되어 있으며, SK하이닉스와 현대차는 주가 상승을 기록했습니다.

- **삼성화재, 항공지연 특약 출시로 고객 편의성 증대*

## 원문 그대로 vs chunking만 사용 비교
> 결론 : chunking된 문서만을 사용하는 게 유리?

[SK하이닉스]

1. 원문 그대로
- **외국인 투자자, 전기·전자 및 운수장비 업종에 집중 매수 및 매도**  <br>
  외국인 투자자들은 한화오션, SK하이닉스, 두산에너빌리티 등을 중점적으로 매수했으며, SK하이닉스, 카카오페이, NAVER 등을 매도했습니다. 특히, 전기·전자 업종에 속한 종목들이 다수 포함되어 있으며, 이들 종목의 주가 변동이 투자 전략에 영향을 미쳤습니다.<br>

- **기관 투자자, 전기·전자 및 운수장비 업종에 집중**  <br>
  기관 투자자들은 삼성전자, 현대차, 삼성전기 등 전기·전자 업종의 종목을 중점적으로 매수했으며, SK하이닉스, KODEX 200선물인버스2X 등을 매도했습니다. <br>이들 종목 중 일부는 전일 대비 주가가 상승했습니다.

- **코스피지수 상승, 투자자 고민 증가**  <br>
  코스피지수가 3200을 넘어섰지만, 주식 매입 시점을 고민하는 투자자들이 증가하고 있습니다. 기존 대장주들의 추가 상승 가능성과 이미 충분히 올랐다는 부담감이 동시에 작용하고 있습니다.

- **주요 종목의 주가 변동**  <br>
  한화오션, SK하이닉스, 두산에너빌리티 등은 전일 대비 주가가 상승했으며, 기아, 삼성전자, 현대차 등은 주가가 하락했습니다. 이러한 변동은 투자자들의 매수 및 매도 전략에 영향을 미치고 있습니다.


2. chunking된 문서만 사용
- **SK하이닉스, 2분기 사상 최대 실적 기록**<br>
  SK하이닉스는 2분기 매출 22조2320억원, 영업이익 9조2129억원을 기록하며 사상 최대 실적을 달성했다. 이는 AI 메모리 수요 증가와 D램 및 낸드플래시 출하량 증가에 기인한다.<br>

- **HBM 시장에서의 경쟁 심화 우려**<br>
  SK하이닉스는 HBM 시장에서의 독점적 지위가 경쟁 심화로 인해 위협받을 수 있다는 우려가 제기되고 있다. 삼성전자와 마이크론의 시장 진입이 예상되며, 가격 하락 가능성이 논의되고 있다.<br>

- **SK하이닉스의 HBM 수요 성장 확신**<br>
  SK하이닉스는 HBM 수요가 지속적으로 증가할 것이라고 확신하며, AI 시장에서의 핵심 제품으로서의 중요성을 강조하고 있다. 이에 따라 HBM 생산을 위한 투자를 확대할 계획이다.

- **주가 변동 및 외부 기관의 평가**<br>
  SK하이닉스의 실적 발표 이후 주가는 일시적으로 상승했으나, 외부 기관의 평가에 따라 변동성을 보였다. 골드만삭스는 HBM 시장의 경쟁 심화로 인한 가격 하락 가능성을 지적하며 투자의견을 하향 조정했다. 반면, 다른 기관들은 SK하이닉스의 기술력과 시장 지배력을 긍정적으로 평가하며, 장기적인 성장 가능성을 강조했다.<br>



# 과거 버전

In [204]:
import pandas as pd
from datetime import datetime, timedelta
# cross-encoder
from sentence_transformers import CrossEncoder
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from summary_function_openai import NewsSummaryAgent, build_summary_graph  # 너가 만든 것

# 1. 모델 준비 (CrossEncoder for Re-ranking) # 예시 모델 하나 생성
rerank_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# 2. 최근 5일치 필터링 함수
def get_recent_articles(df: pd.DataFrame, ticker: str, days: int = 5):
    df['Date'] = pd.to_datetime(df['Date'])
    today = df['Date'].max()
    recent_df = df[
        (df['ticker'] == ticker) &
        (df['Date'] >= today - timedelta(days=days))
    ]
    return recent_df.sort_values(by="Date", ascending=False)

# 3. huggingface 모델 이용.
def rerank_articles(df: pd.DataFrame,ticker:str ,query: str, top_k: int = 5):
    task_df= df[df.ticker == ticker]
    docs = task_df['content'].tolist()
    # 모델 이용?? 
    pairs = [(query, doc) for doc in docs]
    scores = rerank_model.predict(pairs)
    
    task_df_2 = task_df.copy()
    task_df_2['score'] = scores
    return task_df_2.sort_values(by='score', ascending=False).head(top_k)

# 4. 전체 요약 실행 함수
def summarize_top_articles(df: pd.DataFrame, ticker: str, query: str, top_k: int = 5):
    recent_df = get_recent_articles(df, ticker)
    top_df = rerank_articles(recent_df, query=query,ticker=ticker, top_k=top_k)

    agent = NewsSummaryAgent()
    runnable = build_summary_graph(agent)

    results = []
    for _, row in top_df.iterrows():
        state = {
            "article": row['content'],
            "summary": "",
            "feedback": "",
            "iteration": 0
        }
        result = runnable.invoke(state)


        print("✅ 실행 결과 타입:", type(result))
        print("✅ 실행 결과 키 목록:", result.keys())
        print("✅ 실행 결과 전체 내용:", result)

        if "final_summary" not in result:
            print("❌ final_summary 키가 없습니다. 중단합니다.")
            continue  # 또는 raise Exception("final_summary 없음")

        print(f'실행 결과 : {result}')
        results.append({
            "ticker": row['ticker'],
            "date": row['Date'],
            "header": row['header'],
            "url": row['url'],
            "summary": result["final_summary"], 
             "feedback": result.get("last_feedback", "피드백 없음")
        })
    return pd.DataFrame(results)


In [55]:
query = "우리금융지주 관련 시황"
ticker = "우리금융지주"  # 예시
result_df = summarize_top_articles(cusA_news_df, ticker=ticker, query=query, top_k=3)

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


[should_stop] Iteration: 1
[should_stop] Feedback:
 - 정확성: 좋음 <reason> 원문의 내용을 정확하게 반영하고 있음. </reason>
- 포괄성: 부족함 <reason> 원문에서 언급된 '기술 경쟁력 확보, 고객사 확대, 수익구조 개선 등 실질적 사업성과로 뒷받침돼야 한다는 지적' 등의 중요한 내용이 누락되었음. </reason>
- 간결성: 좋음 <reason> 불필요한 표현 없이 요약 내용을 간결하게 전달하였음. </reason>
- 문장구성: 좋음 <reason> 문장이 자연스럽고 명확하게 구성되어 있음. </reason>

[피드백]
요약의 포괄성이 부족한 점이 아쉽습니다. 원문에서 언급된 '기술 경쟁력 확보, 고객사 확대, 수익구조 개선 등 실질적 사업성과로 뒷받침돼야 한다는 지적' 등의 중요한 내용을 요약에 포함시키면 더욱 완벽한 요약이 될 것 같습니다. 이 부분을 고려하여 요약을 수정해보시는 것을 추천드립니다.
✅ '정확성' 평가 통과
❌ '포괄성' 평가에서 좋음이 아님
[should_stop] next_step = no


KeyError: 'Input to PromptTemplate is missing variables {\'"foo"\', \'"properties"\'}.  Expected: [\'"foo"\', \'"properties"\', \'article\', \'feedback\', \'summary\'] Received: [\'article\', \'summary\', \'feedback\']\nNote: if you intended {"foo"} to be part of the string and not a variable, please escape it with double curly braces like: \'{{"foo"}}\'.\nFor troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_PROMPT_INPUT '

In [35]:
def summarize_top_articles(total_contents :str):

    agent = NewsSummaryAgent()
    runnable = build_summary_graph(agent)
    results = []
    
    doc = total_contents

    state = {
        "article": doc['content'],
        "summary": "",
        "feedback": "",
        "iteration": 0
    }
    result = runnable.invoke(state)


    print("✅ 실행 결과 타입:", type(result))
    print("✅ 실행 결과 키 목록:", result.keys())
    print("✅ 실행 결과 전체 내용:", result)

    if "final_summary" not in result:
        print("❌ final_summary 키가 없습니다.")

    print(f'실행 결과 : {result}')
    results.append({
        "ticker": doc['ticker'],
        "date": doc['Date'],
        "header": doc['title'],
        "url": doc['url'],
        "summary": result["final_summary"], 
            "feedback": result.get("last_feedback", "피드백 없음")
    })
    return pd.DataFrame(results)
